In [ ]:
from pathlib import Path
import sys

# Localizar la raiz del repositorio desde Jupyter o terminal
RAIZ_REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "notebooks").is_dir() and (p / "datos").is_dir()),
    None
)
if RAIZ_REPO is None:
    raise FileNotFoundError("Ejecuta desde el repositorio caso-itam")

sys.path.insert(0, str(RAIZ_REPO / "src"))


# Validación del resultado nulo territorial y resultados

En este cuaderno pongo a prueba el hallazgo central del proyecto: la probabilidad de que un delito llegue a carpeta de investigación no varía de forma detectable entre alcaldías. Primero mido la potencia del método para detectar esa variación si existiera. Después verifico la medición con un control positivo, busco la heterogeneidad donde sí está (el tipo de delito), corrijo el mapa de carpetas y reviso la robustez de todo lo anterior.

Uso como insumos fijos las estimaciones finales de d, k y R, la comparación de modelos y la tabla maestra por alcaldía. Aquí no reestimo nada de eso.

## 1. Insumos

In [ ]:
# Carga e inspección de insumos

import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

RAIZ = RAIZ_REPO
assert RAIZ.exists(), f"No existe {RAIZ}; ajusta la ruta"

archivos = {
    "estimaciones":   "estimaciones_finales_benchmark.parquet",  # d, k, R finales por alcaldía-año
    "modelos":        "modelos_comparacion.parquet",             # FH / FH+alcaldía / Rao-Yu
    "incidentes":     "envipe_incidentes.parquet",               # microdatos de incidentes ENVIPE
    "maestro":        "maestro_alcaldia.parquet",                # tabla maestra 16 × 5 (versión final, no v0–v3)
    "carpetas":       "C_alcaldia_anio.parquet",                 # carpetas por alcaldía-año
    "completitud":    "factor_completitud.parquet",              # corrección por rezago 2024
    "consistencia":   "consistencia_R_dk.csv",                   # R vs d·k
    "tabla_metodos":  "tabla_comparativa_metodos.csv",
}

rutas, datos = {}, {}
for nombre, archivo in archivos.items():
    # rglob busca en todas las subcarpetas; el nombre exacto evita confundir
    # maestro_alcaldia.parquet con maestro_alcaldia_v1.parquet, etc.
    encontrados = sorted(RAIZ.rglob(archivo), key=lambda p: str(p))
    if not encontrados:
        print(f"[FALTA]  {archivo}")
        continue
    if len(encontrados) > 1:
        print(f"[AVISO]  {archivo} aparece {len(encontrados)} veces; uso la más reciente:")
        for p in encontrados:
            print(f"           {p.relative_to(RAIZ)}")
    rutas[nombre] = encontrados[0]
    ruta = encontrados[0]
    datos[nombre] = pd.read_parquet(ruta) if ruta.suffix == ".parquet" else pd.read_csv(ruta)

print("\nRutas usadas:")
for nombre, ruta in rutas.items():
    print(f"  {nombre:14s} -> {ruta.relative_to(RAIZ)}")

# Resumen de cada tabla: dimensiones, columnas, tipos y primeras filas
for nombre, df in datos.items():
    print("=" * 90)
    print(f"{nombre}  ->  {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print("-" * 90)
    print(df.dtypes.to_string())
    print("\nPrimeras filas:")
    print(df.head(3).to_string())

## 2. Potencia: ¿qué tan grande tendría que ser una diferencia para detectarla?

Un resultado nulo solo es informativo si el método podía encontrar lo que no encontró. Mido esa capacidad de tres formas: con un motor de estimación que reproduce mi modelo de componentes de varianza, con la diferencia mínima detectable analítica y con una simulación en la que inyecto diferencias territoriales de tamaño conocido.

### 2.1 Motor REML de dos componentes

In [ ]:
# Insumos de la potencia y motor REML de dos componentes

# EL MODELO CONTRA EL QUE MIDO LA POTENCIA (sin covariables, solo años)
#   θ_at = μ_t + u_a + v_at + e_at
#     u_a  ~ N(0, σ²_alc)   persistente: el mismo los 5 años  ← lo que busco
#     v_at ~ N(0, σ²_cel)   variación de año a año dentro de la alcaldía
#     e_at ~ N(0, ψ_at)     error muestral, ψ conocida (var_theta_asen)
#
# ¿Por qué dos componentes y no solo u_a? Porque al seleccionar modelos
# encontré que la variación de año a año existe (exceso de 32% a 56% sobre ψ). Si el modelo
# de prueba la ignora, la confunde con señal persistente y la prueba "detecta"
# de más. Con v_at incluido, la prueba solo responde a lo que se repite.
#
# LA PRUEBA: razón de verosimilitudes (REML) de H0: σ²_alc = 0.
#   LRT = 2·(ℓ_completo − ℓ_nulo). Como σ²_alc ≥ 0, bajo H0 el parámetro está
#   en la frontera y la nula es ½χ²₀ + ½χ²₁. Su cuantil 0.95 es 2.706
#   (el 0.90 de χ²₁). Es la misma convención de mi cota por verosimilitud perfil.
#
# VERIFICACIÓN: escribí este motor para este análisis, así que antes de usarlo
# compruebo que reproduce el modelo de dos componentes sin covariables que
# estimé al seleccionar modelos: parte persistente 19% en d (p = 0.29), 0% en k
# y 12% en R (p = 0.36). Si no los reproduce, no sigo.
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import chi2, norm

RUTA_SALIDA = RAIZ / "resultados" / "dia13_14"
RUTA_SALIDA.mkdir(exist_ok=True)

# Directos en escala θ = arcsen√p, con su ψ. En el archivo de modelos, para
# el método directo, ecm_theta es exactamente la ψ suavizada que construí para los modelos de área.
directos = (datos["modelos"].query("metodo == '1_directo'")
            .sort_values(["param", "alc", "anio_ref"]).reset_index(drop=True))

def preparar(p):
    """Datos de un parámetro: celdas con θ observada, X de efectos de año,
    θ, ψ y la matriz ZZ' (1 si dos celdas son de la misma alcaldía)."""
    df = directos[(directos["param"] == p) & directos["theta"].notna()].reset_index(drop=True)
    anios = sorted(df["anio_ref"].unique())
    X = np.column_stack([np.ones(len(df))] +
                        [(df["anio_ref"] == a).to_numpy(float) for a in anios[1:]])
    Z = pd.get_dummies(df["alc"], dtype=float).to_numpy()
    return df, X, df["theta"].to_numpy(), df["ecm_theta"].to_numpy(), Z @ Z.T

def reml_ll(y, X, psi, ZZ, s_alc, s_cel):
    """Log-verosimilitud REML con V = diag(ψ + σ²_cel) + σ²_alc·ZZ'."""
    V = np.diag(psi + s_cel) + s_alc * ZZ
    L = np.linalg.cholesky(V)                       # V = LL', estable y rápido
    Li = np.linalg.inv(L); Vi = Li.T @ Li
    XtVi = X.T @ Vi; XVX = XtVi @ X
    P = Vi - XtVi.T @ np.linalg.solve(XVX, XtVi)    # proyector de REML
    return -0.5 * (2 * np.log(np.diag(L)).sum() + np.linalg.slogdet(XVX)[1] + y @ P @ y)

def ajustar_dos_componentes(y, X, psi, ZZ, arranques=((1e-4, 1e-4), (1e-3, 1e-3), (1e-5, 3e-4))):
    """Ajusta el modelo nulo (σ²_alc = 0) y el completo; devuelve la prueba."""
    nulo = minimize_scalar(lambda c: -reml_ll(y, X, psi, ZZ, 0.0, c),
                           bounds=(0, 0.05), method="bounded", options={"xatol": 1e-10})
    mejor = None
    for s0 in [*arranques, (1e-4, max(nulo.x, 1e-6))]:   # varios arranques: la superficie es plana
        r = minimize(lambda s: -reml_ll(y, X, psi, ZZ, s[0], s[1]), s0,
                     method="L-BFGS-B", bounds=[(0, 0.05), (0, 0.05)])
        if mejor is None or r.fun < mejor.fun:
            mejor = r
    ll_comp, ll_nulo = max(-mejor.fun, -nulo.fun), -nulo.fun   # el completo anida al nulo
    lrt = max(2 * (ll_comp - ll_nulo), 0.0)
    return {"s2_alc": mejor.x[0], "s2_cel": mejor.x[1], "s2_cel_nulo": nulo.x,
            "lrt": lrt, "p_valor": 0.5 * chi2.sf(lrt, 1) if lrt > 0 else 1.0}

# --- Verificación contra la selección de modelos ------------------------------------------
ref_seleccion = {"d": (0.19, 0.29), "k": (0.00, 1.00), "R": (0.12, 0.36)}
filas = []
for p in ["d", "k", "R"]:
    df, X, y, psi, ZZ = preparar(p)
    r = ajustar_dos_componentes(y, X, psi, ZZ)
    frac = r["s2_alc"] / (r["s2_alc"] + r["s2_cel"])
    filas.append({"param": p, "celdas": len(y), "s2_alc": r["s2_alc"], "s2_cel": r["s2_cel"],
                  "frac_persistente": frac, "lrt": r["lrt"], "p_valor": r["p_valor"],
                  "ref_frac": ref_seleccion[p][0], "ref_p": ref_seleccion[p][1]})
    assert abs(frac - ref_seleccion[p][0]) < 0.01 and abs(r["p_valor"] - ref_seleccion[p][1]) < 0.01, \
        f"{p}: el motor no reproduce la selección de modelos"

verif = pd.DataFrame(filas)
print("Modelo alcaldía + celda, sin covariables (solo efectos de año):\n")
print(verif.round(6).to_string(index=False))
print("\nOK: el motor reproduce la selección de modelos en los tres parámetros.")

### 2.2 Diferencia mínima detectable

In [ ]:
# Diferencia mínima detectable (DMD), analítica

# DMD = (z_0.975 + z_0.80) · EE(comparación) ≈ 2.80 · EE
#
# Busco diferencias PERSISTENTES (las que se repiten los 5 años). Para esa
# pregunta, la variación de año a año (σ²_cel) es ruido igual que el muestral.
#
# Cuatro situaciones, de la más débil a la más fuerte:
#   horizonte "un_anio":       una sola celda por alcaldía
#       Var(θ_at) = ψ_at + σ²_cel
#   horizonte "media_5_anios": promedio simple de las 5 celdas
#       Var(θ̄_a) = Σ_t ψ_at / 25 + σ²_cel / 5
#   comparación "par_de_alcaldias":   EE = √(Var_a + Var_b)
#   comparación "alcaldia_vs_ciudad": EE = √Var_a  (la ciudad casi no tiene error)
#
# Para cada caso reporto la mediana sobre alcaldías (o sobre los 120 pares),
# y también el mejor y el peor caso.
#
# TRADUCCIÓN A PROPORCIÓN: centro la DMD en el nivel de ciudad (promedio
# 2020–2024, P_CIUDAD, mis cifras de ciudad): θ_alta = θ_c + DMD/2 y
# θ_baja = θ_c − DMD/2. Después regreso con p = sen²θ y reporto la brecha en
# puntos y la razón alta/baja, que es la escala del 3.5 a 1.

import itertools

P_CIUDAD = {"d": 0.1098, "k": 0.6644, "R": 0.0728}
Z_POT = norm.ppf(0.975) + norm.ppf(0.80)          # ≈ 2.80
S2_CEL = verif.set_index("param")["s2_cel"].to_dict()

def dmd_a_proporcion(p, dmd):
    """Centra la DMD (escala θ) en el nivel de ciudad y la regresa a proporción."""
    tc = np.arcsin(np.sqrt(P_CIUDAD[p]))
    alta, baja = min(tc + dmd / 2, np.pi / 2), max(tc - dmd / 2, 0.0)
    return np.sin(alta) ** 2, np.sin(baja) ** 2

filas_dmd = []
for p in ["d", "k", "R"]:
    df, X, y, psi, ZZ = preparar(p)
    g = df.assign(psi=psi).groupby("alc")["psi"]
    var_horizonte = {
        "un_anio":       g.median() + S2_CEL[p],                         # celda típica
        "media_5_anios": g.sum() / g.size() ** 2 + S2_CEL[p] / g.size(), # k: Milpa Alta tiene 4
    }
    for horizonte, v in var_horizonte.items():
        ee = {"par_de_alcaldias":   np.array([np.sqrt(v[a] + v[b])
                                              for a, b in itertools.combinations(v.index, 2)]),
              "alcaldia_vs_ciudad": np.sqrt(v.to_numpy())}
        for comparacion, e in ee.items():
            for caso, q in [("mediana", 0.5), ("mejor", 0.0), ("peor", 1.0)]:
                dmd = Z_POT * np.quantile(e, q)
                alta, baja = dmd_a_proporcion(p, dmd)
                filas_dmd.append({"param": p, "horizonte": horizonte, "comparacion": comparacion,
                                  "caso": caso, "dmd_theta": dmd, "p_alta": alta, "p_baja": baja,
                                  "brecha_pp": 100 * (alta - baja),
                                  "razon_alta_baja": alta / baja if baja > 0 else np.inf})

tabla_dmd = pd.DataFrame(filas_dmd)

print("Diferencia mínima detectable (80% de potencia, α = 0.05), caso MEDIANO:\n")
print(tabla_dmd.query("caso == 'mediana'")
      .drop(columns="caso").round(3).to_string(index=False))
print("\nRango mejor–peor de la razón alta/baja:\n")
print(tabla_dmd.pivot_table(index=["param", "horizonte", "comparacion"], columns="caso",
                            values="razon_alta_baja")[["mejor", "mediana", "peor"]].round(2).to_string())

**Lo que leo en esta tabla.**

1. **Con un solo año, la encuesta no distingue ni una brecha de 3.5 a 1.** Para separar dos alcaldías en un año, la diferencia en R tendría que ser de 4.9 a 1 en el par mediano y de hasta 29 a 1 en el peor; contra la ciudad, de 2.9 a 1. Por eso no uso como evidencia el criterio de que ninguna alcaldía tiene un intervalo anual que no se traslape con la mediana: es una comparación de un solo año, sin potencia. Además se calcula sobre estimaciones modeladas, que con σ²_u chica se encogen hacia la media por construcción. Lo mismo vale para la razón máx/mín de 1.41 a 1.73 de las estimaciones modeladas.
2. **La potencia está en la persistencia.** Con el promedio de cinco años, la DMD de R baja a 1.6 a 1 contra la ciudad y a 2.0 a 1 entre pares. Coincide con mi cota superior por verosimilitud perfil (alrededor de 2 a 1): es la misma información vista desde el otro lado.
3. **En k la razón engaña.** Una razón de 1.3 a 1.5 parece chica, pero alrededor de 66% equivale a 18 a 26 puntos porcentuales.
4. **Lo que falta.** Esta tabla se refiere a una comparación aislada. La prueba de fondo usa las 16 alcaldías y los cinco años a la vez, y debería tener más potencia ante un gradiente repartido entre muchas alcaldías. Como esa potencia no tiene fórmula cerrada, la simulo.

### 2.3 Simulación: diferencias territoriales inyectadas

In [ ]:
# Simulación: inyectar diferencias territoriales y ver si el método las ve

# LA LÓGICA
#   La diferencia mínima detectable contesta "¿qué tan grande debe ser UNA comparación?". Pero
#   la prueba de fondo del resultado nulo no compara pares: pregunta si las 16
#   alcaldías, juntas y durante 5 años, varían más de lo que el ruido explica.
#   Esa prueba (LRT de σ²_alc = 0) usa toda la información a la vez, y su
#   potencia no tiene fórmula cerrada. Por eso la simulo:
#     1. Construyo un "mundo verdadero" donde SÍ hay diferencias persistentes
#        de un tamaño conocido.
#     2. Genero datos falsos de ese mundo con el MISMO ruido que tienen mis
#        datos: los mismos ψ_at y la misma variación de año a año σ²_cel.
#     3. Corro la prueba. Repito B veces. Potencia = fracción de rechazos.
#
# EL MUNDO VERDADERO: p_at = p_t · (C_a / C̄)^λ
#   p_t   = nivel de ciudad de cada año (media GLS de mis directos)
#   C_a   = tasa de carpetas por habitante de la alcaldía (media geométrica
#           2020–2024), C̄ su media geométrica
#   λ     = qué fracción (en escala logarítmica) de la desigualdad de las
#           carpetas se traslada al parámetro
#   λ = 0 → nadie difiere (sirve para verificar que la prueba rechaza ~5%)
#   λ = 1 → el parámetro varía exactamente 3.5 a 1, con el MISMO orden que
#           las carpetas. Es el contrafactual de que "toda la
#           desigualdad del mapa viene de la propensión a llegar a carpeta".
#   La razón máx/mín inyectada es 3.54^λ.
#
# DOS PATRONES
#   "carpetas":  las alcaldías altas son las de más carpetas (el contrafactual)
#   "aleatorio": la misma dispersión, pero barajando qué alcaldía es alta en
#                cada réplica. Controla que la potencia no dependa de que las
#                alcaldías de muchas carpetas tengan casualmente ψ chicas.
#
# En k, multiplicar 0.66 por más de ~1.5 rebasa 1; esos escenarios se marcan
# "imposibles" y no se simulan. Por eso para k la brecha en PUNTOS es la
# métrica útil, no la razón.
#
# Tiempo: ~25 ms por ajuste → con B = 300, unos 8 a 12 minutos en total.
import time

B_REPS  = 300
LAMBDAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.75, 1.0]
CRITICO = 2.706                                   # 0.95 de ½χ²₀ + ½χ²₁
rng = np.random.default_rng(20261013)             # semilla: reproducible

# --- Patrón de las carpetas ---------------------------------------------------
C_media = (datos["maestro"].groupby("cve_mun")["tasa18_nucleo"]
           .apply(lambda s: np.exp(np.log(s).mean())))
patron_C = np.log(C_media / np.exp(np.log(C_media).mean()))   # log(C_a / C̄)
RAZON_C = float(np.exp(patron_C.max() - patron_C.min()))
print(f"Razón máx/mín de la tasa de carpetas (media geométrica 2020–2024): {RAZON_C:.2f}\n")

filas_sim = []
for p in ["d", "k", "R"]:
    df, X, y, psi, ZZ = preparar(p)
    alc_celda = df["alc"].to_numpy()
    s2_cel = S2_CEL[p]

    # Nivel de cada año: media GLS bajo el modelo sin efecto de alcaldía
    Vi0 = np.diag(1 / (psi + s2_cel))
    mu = X @ np.linalg.solve(X.T @ Vi0 @ X, X.T @ Vi0 @ y)
    p_anio = np.sin(mu) ** 2

    for lam in LAMBDAS:
        for nombre_patron in ["carpetas", "aleatorio"]:
            t0 = time.time()
            rechazos, s2_est, s2_real, brechas = 0, [], [], []
            imposible = False
            for b in range(B_REPS):
                pat = patron_C if nombre_patron == "carpetas" else \
                      pd.Series(rng.permutation(patron_C.to_numpy()), index=patron_C.index)
                # Mundo verdadero en proporción, luego a θ
                p_verd = p_anio * np.exp(lam * pat.reindex(alc_celda).to_numpy())
                if p_verd.max() >= 0.98:
                    imposible = True
                    break
                u = np.arcsin(np.sqrt(p_verd)) - mu        # efecto persistente en θ
                # Datos simulados: verdad + variación de año a año + error muestral
                y_sim = (mu + u
                         + rng.normal(0, np.sqrt(s2_cel), len(y))
                         + rng.normal(0, np.sqrt(psi)))
                r = ajustar_dos_componentes(y_sim, X, psi, ZZ, arranques=((1e-4, 1e-4),))
                rechazos += r["lrt"] > CRITICO
                s2_est.append(r["s2_alc"])
                s2_real.append(pd.Series(u).groupby(alc_celda).mean().var(ddof=0))
                p_alc = pd.Series(p_verd).groupby(alc_celda).mean()
                brechas.append(100 * (p_alc.max() - p_alc.min()))

            if imposible:
                print(f"{p}  {nombre_patron:9s}  razón {RAZON_C**lam:4.2f}  imposible: el parámetro superaría 0.98")
                continue
            pot = rechazos / B_REPS
            filas_sim.append({"param": p, "patron": nombre_patron, "lambda": lam,
                              "razon_inyectada": RAZON_C ** lam,
                              "brecha_pp_inyectada": np.mean(brechas),
                              "potencia": pot,
                              "ee_potencia": np.sqrt(pot * (1 - pot) / B_REPS),
                              "s2_alc_inyectada": np.mean(s2_real),
                              "s2_alc_estimada_mediana": np.median(s2_est)})
            print(f"{p}  {nombre_patron:9s}  razón {RAZON_C**lam:4.2f}  "
                  f"brecha {np.mean(brechas):5.1f} pp  potencia {pot:5.3f}   ({time.time()-t0:4.1f} s)")

tabla_sim = pd.DataFrame(filas_sim)

print("\nPotencia por razón inyectada (filas) y patrón (columnas):\n")
print(tabla_sim.pivot_table(index=["param", "razon_inyectada"], columns="patron",
                            values="potencia").round(3).to_string())

**Lo que leo en la simulación.**

La prueba está bien calibrada: sin diferencias inyectadas rechaza entre 2.7% y 5.7% de las veces, alrededor del 5% nominal y ligeramente conservadora en d y R, como es habitual en una prueba de razón de verosimilitudes en la frontera. Por eso el resto de la tabla se puede leer como potencia.

Lo que puedo afirmar:

- **El contrafactual queda descartado.** Si R variara 3.5 a 1 con el orden de las carpetas, la prueba lo habría detectado en 300 de 300 réplicas, y en 99% con el patrón barajado. La ausencia de un gradiente de esa magnitud en R no se explica por falta de potencia.
- **El método no es ciego.** Recupera gradientes de 2.6 a 1 casi siempre en d y R. Esto funciona como un control positivo por simulación, independiente del control con robo de vehículo.

Lo que no puedo afirmar:

- **Diferencias moderadas pasan inadvertidas.** El 80% de potencia en R se alcanza alrededor de 2.1 a 1 con el patrón de carpetas y de 2.4 a 1 con el barajado; una brecha de 1.5 a 1 se detecta solo 15% a 21% de las veces. Decir que R es igual en las 16 alcaldías sería excesivo. Lo correcto es que R no varía más de alrededor de 2 a 1, lo que coincide con la cota por verosimilitud perfil.
- **El efecto de R sobre el mapa es parcial, no nulo.** En escala logarítmica, 2.1 a 1 es alrededor del 60% de la brecha de 3.5 a 1. Por eso afirmo que la desigualdad del mapa no se explica *principalmente* por la propensión a llegar a carpeta, no que no se explica en absoluto. La estimación puntual (12% persistente, p = 0.36) apunta a mucho menos, pero la potencia no permite descartar una contribución parcial.
- **En k no hay potencia.** Para 80% hace falta una brecha de alrededor de 25 puntos porcentuales, del mismo orden que mi cota de 19.5 puntos. Lo que sostengo sobre k es su nivel (66%, lejos del 100% que exige la ley), no su homogeneidad.

El patrón de carpetas tiene algo más de potencia que el barajado porque las alcaldías con más carpetas tienen muestras grandes y ψ chicas. Cito el barajado, que es la cifra conservadora.

### 2.4 Umbrales de potencia y entregable

In [ ]:
# Umbrales de potencia 

# UMBRAL: la razón (y la brecha en puntos) a partir de la cual la prueba
# detecta la diferencia con 50% y con 80% de probabilidad. Se obtiene
# interpolando la curva de potencia en escala log(razón), que es la escala en
# la que construí los escenarios (λ).
#
# La curva simulada tiene ruido (EE ≈ 0.02–0.03 con B = 300) y puede bajar un
# poco entre puntos vecinos. Antes de interpolar la hago no decreciente con un
# máximo acumulado; así la interpolación está bien definida.
#
# El CSV apila tres bloques con una columna "bloque":
#   dmd_analitica: diferencia mínima detectable analítica
#   simulacion:    curva de potencia simulada
#   umbral:        este resumen (lo que cito en el escrito)
def umbral(g, nivel):
    """Razón y brecha en pp a la que la potencia alcanza 'nivel'."""
    g = g.sort_values("razon_inyectada")
    pot = np.maximum.accumulate(g["potencia"].to_numpy())
    if pot.max() < nivel:
        return np.nan, np.nan                      # no se alcanza en la malla
    log_r = np.interp(nivel, pot, np.log(g["razon_inyectada"]))
    brecha = np.interp(log_r, np.log(g["razon_inyectada"]), g["brecha_pp_inyectada"])
    return np.exp(log_r), brecha

filas_u = []
for (p, patron), g in tabla_sim.groupby(["param", "patron"]):
    r50, b50 = umbral(g, 0.50)
    r80, b80 = umbral(g, 0.80)
    pot_contraf = g.loc[np.isclose(g["lambda"], 1.0), "potencia"]
    filas_u.append({"param": p, "patron": patron,
                    "razon_pot50": r50, "brecha_pp_pot50": b50,
                    "razon_pot80": r80, "brecha_pp_pot80": b80,
                    # fracción de la brecha de carpetas (en log) detectable al 80%
                    "frac_log_brecha_C_pot80": np.log(r80) / np.log(RAZON_C),
                    "potencia_contrafactual_3p5": pot_contraf.iloc[0] if len(pot_contraf) else np.nan,
                    "tamano_prueba": g.loc[g["lambda"] == 0, "potencia"].iloc[0]})
tabla_umbral = pd.DataFrame(filas_u)

print("Umbrales de detección de la prueba de varianza persistente (16 alcaldías × 5 años):\n")
print(tabla_umbral.round(3).to_string(index=False))

# --- Entregable --------------------------------------------------------------
validacion = pd.concat([tabla_dmd.assign(bloque="dmd_analitica"),
                        tabla_sim.assign(bloque="simulacion"),
                        tabla_umbral.assign(bloque="umbral")], ignore_index=True)
validacion.insert(0, "bloque", validacion.pop("bloque"))
validacion.to_csv(RUTA_SALIDA / "validacion_potencia.csv", index=False, encoding="utf-8-sig")
print(f"\nGuardado: {RUTA_SALIDA / 'validacion_potencia.csv'}  ({len(validacion)} filas)")

**Síntesis.** La prueba de varianza persistente, calibrada por simulación (tamaño empírico de 0.03 a 0.06), habría detectado con probabilidad de 0.99 a 1.00 que R reprodujera la desigualdad de 3.5 a 1 de las carpetas. Su umbral de 80% de potencia está entre 2.1 y 2.4 a 1, de modo que los datos son compatibles con diferencias moderadas en R, pero no con que R explique la mayor parte de la desigualdad del registro. Para k, el umbral equivale a una brecha de 26 a 32 puntos porcentuales: la encuesta no tiene potencia para evaluar diferencias territoriales en la barrera institucional.

## 3. Estimador directo con varianza de diseño

Para el control positivo y para la heterogeneidad por tipo de delito necesito estimaciones directas a nivel ciudad con su varianza de diseño. Escribo el estimador una vez y lo verifico contra mis cifras de ciudad antes de usarlo.

In [ ]:
# Estimador directo de razón con linealización de Taylor

# PARA QUÉ: d, k y R por tipo de delito a nivel ciudad. Ahí
# la muestra es grande y no hace falta modelo de área: basta el directo con su
# varianza de diseño.
#
# LOS TRES PARÁMETROS SON RAZONES:  r = Σ w·y / Σ w·x
#   d: y = denunció,  x = 1 (todo incidente)
#   k: y = carpeta,   x = denunció
#   R: y = carpeta,   x = 1
#
# VARIANZA POR LINEALIZACIÓN (Taylor)
#   La razón no es lineal, pero alrededor de su valor se comporta como el total
#   de la variable linealizada  z_i = w_i (y_i − r·x_i) / X̂.
#   Var(r̂) ≈ Σ_h  n_h/(n_h − 1) · Σ_j (z_hj − z̄_h)²
#     h = estrato de diseño (EST_DIS), j = UPM (UPM_DIS), z_hj = suma de z en
#     la UPM. Es el estimador "con reemplazo en la primera etapa", el estándar
#     para la ENVIPE.
#
# ESTIMACIÓN DE DOMINIO: el dominio (CDMX, un tipo de delito) NO es un estrato.
# Se usan TODAS las UPM del año y se pone z = 0 fuera del dominio. Si
# filtrara primero, perdería la variación en el tamaño del dominio y
# subestimaría la varianza.
#   Limitación: el archivo solo trae UPM con al menos un incidente. Las UPM sin
#   incidentes aportan z = 0 y faltan en n_h. Su efecto es despreciable, porque
#   agregarlas sube n_h/(n_h−1) apenas y no cambia Σ(z − z̄)² de forma material.
#
# ESTRATOS POR AÑO: cada levantamiento es una muestra independiente, así que el
# estrato se define como año + EST_DIS. Eso permite agrupar los 5 años.
#
# FACTOR: FAC_DEL original, sin calibrar a CONAPO. A nivel ciudad reproduce las
# cifras del INEGI, que es la referencia natural para las cifras por tipo. La
# calibración (FAC_DEL_AJ) solo importa para repartir entre alcaldías.
#
# VERIFICACIÓN: debe reproducir mis cifras de ciudad (~11 / ~66 / ~7) y sus
# errores estándar (≈ 0.9 pp en d, 3 a 4 pp en k).
inc = datos["incidentes"].copy()
inc["anio_ref"]   = inc["anio_envipe"] - 1
inc["estrato"]    = inc["anio_envipe"].astype(str) + "_" + inc["EST_DIS"]
inc["upm_diseno"] = inc["estrato"] + "_" + inc["UPM_DIS"]
inc["todos"]      = True

# Universo: ocurrido en la CDMX, sin vandalismo. Incluye los incidentes sin
# alcaldía (999): a nivel ciudad sí cuentan.
UNIVERSO = inc["cdmx_ocu"] & ~inc["excl_vandalismo"]

DEFINICION = {"d": ("denuncio", "todos"),
              "k": ("carpeta",  "denuncio"),
              "R": ("carpeta",  "todos")}

def razon_taylor(df, dominio, num, den, peso="FAC_DEL"):
    """Razón ponderada Σw·y/Σw·x en un dominio, con EE por linealización."""
    w = df[peso].to_numpy()
    m = dominio.to_numpy()
    y = (df[num].to_numpy() & m).astype(float)     # fuera del dominio → 0
    x = (df[den].to_numpy() & m).astype(float)
    Y, X = (w * y).sum(), (w * x).sum()
    r = Y / X
    z = w * (y - r * x) / X                        # variable linealizada
    tot = (pd.DataFrame({"estrato": df["estrato"].to_numpy(),
                         "upm": df["upm_diseno"].to_numpy(), "z": z})
           .groupby(["estrato", "upm"], sort=False)["z"].sum().reset_index())
    g = tot.groupby("estrato")["z"]
    n_h = g.transform("size")
    dev2 = (tot["z"] - g.transform("mean")) ** 2
    ok = n_h > 1                                   # estrato con 1 UPM: sin aporte
    var = (dev2[ok] * n_h[ok] / (n_h[ok] - 1)).sum()
    return {"est": r, "ee": np.sqrt(var), "n": int((m & (x > 0)).sum())}

def estimar(df, dominio, peso="FAC_DEL"):
    """d, k y R con su EE y su n muestral en un dominio."""
    fila = {}
    for p, (num, den) in DEFINICION.items():
        r = razon_taylor(df, dominio, num, den, peso)
        fila.update({p: r["est"], f"ee_{p}": r["ee"], f"n_{p}": r["n"]})
    return fila

# --- Verificación: cifras de ciudad por año ------------------------------------
ciudad = pd.DataFrame([{"anio_ref": a, **estimar(g, UNIVERSO.loc[g.index])}
                       for a, g in inc.groupby("anio_ref")])
en_pp = {c: 100 * ciudad[c] for c in ["d", "ee_d", "k", "ee_k", "R", "ee_R"]}
print("Cifras de ciudad por año (FAC_DEL, ocurrencia CDMX, sin vandalismo), en %:\n")
print(ciudad.assign(**en_pp).round(2).to_string(index=False))

## 4. Control positivo

El robo total de vehículo es mi control positivo: denunciar es requisito para cobrar el seguro, así que su d debería ser alta y pareja entre alcaldías. Como contraste uso el tipo de delito con la d más baja, elegido con una regla que fijé antes de ver los resultados.

### 4.1 Los dos extremos frente al resto de los delitos

In [ ]:
# Control positivo: d, k y R por tipo de delito (ciudad, 2020–2024)

# QUÉ VALIDA, Y QUÉ NO
#   Valida la MEDICIÓN: si los denominadores o la construcción de d estuvieran
#   mal, el robo de vehículo, donde la denuncia es requisito para cobrar el
#   seguro, no saldría alto. Si el estimador estuviera "aplastado" hacia la
#   media, no separaría tipos de delito que sabemos muy distintos.
#   NO valida la potencia TERRITORIAL: eso lo mide la simulación de potencia.
#   Aquí la muestra por tipo a nivel ciudad es mucho mayor que por alcaldía,
#   así que "detecta diferencias entre tipos" no implica "detectaría las mismas
#   diferencias entre alcaldías". Así lo escribo.
#
# EL EXTREMO OPUESTO, CON UNA REGLA FIJADA ANTES DE MIRAR
#   El tipo de d más bajo entre los que tienen n ≥ 100 incidentes y
#   confiabilidad alta o tolerable en d. Fijar la regla evita escoger el
#   contraste más vistoso después de ver la tabla.
#
# CONTRASTE CONTRA EL RESTO
#   tipo y "resto de los delitos" son dominios de la MISMA muestra: comparten
#   UPM, así que no son independientes. La varianza de la diferencia se obtiene
#   linealizando la diferencia (z_tipo − z_resto) y pasándola por la misma
#   fórmula de diseño. Así la covarianza entra sola.
#
# CONFIABILIDAD: el mismo criterio del INEGI que uso para los estimadores
# directos por alcaldía: CV < 15% alta, 15–30% tolerable, > 30% con reservas.
# Degenerada cuando el EE o la estimación valen 0 (CV nulo o indefinido).

ETIQUETAS = {
    "01": "Robo total de vehículo",
    "02": "Robo de accesorios o autopartes",
    "04": "Robo a casa habitación",
    "05": "Robo o asalto en calle o transporte",
    "06": "Otro tipo de robo",
    "07": "Fraude bancario",
    "08": "Fraude al consumidor",
    "09": "Extorsión",
    "10": "Amenazas verbales",
    "11": "Lesiones",
    "12": "Secuestro",
    "13": "Hostigamiento o abuso sexual",
    "14": "Violación sexual",
    "15": "Otros delitos",
}

def confiabilidad(est, ee, n):
    """Criterio INEGI: CV < 15% alta | 15–30% tolerable | > 30% con reservas.
    Degenerada: EE = 0 o estimación = 0 (CV nulo o indefinido)."""
    if pd.isna(est) or n == 0:
        return "sin estimación"
    if ee == 0 or est == 0:
        return "degenerada"
    cv = 100 * ee / est
    if cv < 15:
        return "alta"
    if cv <= 30:
        return "tolerable"
    return "con reservas"

# --- 1. Los tres parámetros por tipo, cinco años agrupados ----------------------
filas = []
for cod in sorted(inc.loc[UNIVERSO, "BPCOD"].unique()):
    dom = UNIVERSO & (inc["BPCOD"] == cod)
    filas.append({"BPCOD": cod, "tipo": ETIQUETAS.get(cod, cod), **estimar(inc, dom)})
por_tipo = pd.DataFrame(filas)
for p in ["d", "k", "R"]:
    por_tipo[f"cv_{p}"]   = por_tipo[f"ee_{p}"] / por_tipo[p]
    por_tipo[f"conf_{p}"] = [confiabilidad(e, s, n) for e, s, n in
                             zip(por_tipo[p], por_tipo[f"ee_{p}"], por_tipo[f"n_{p}"])]
    por_tipo[f"ic_inf_{p}"] = (por_tipo[p] - 1.96 * por_tipo[f"ee_{p}"]).clip(lower=0)
    por_tipo[f"ic_sup_{p}"] = (por_tipo[p] + 1.96 * por_tipo[f"ee_{p}"]).clip(upper=1)

cols = ["BPCOD", "tipo", "n_d", "d", "ee_d", "conf_d", "n_k", "k", "ee_k", "conf_k"]
vista = por_tipo[cols].copy()
for c in ["d", "ee_d", "k", "ee_k"]:
    vista[c] = 100 * vista[c]
print("d y k por tipo de delito, CDMX 2020–2024 agrupado (%), orden por d:\n")
print(vista.sort_values("d", ascending=False).round(2).to_string(index=False))

# --- 2. El extremo opuesto, con la regla fijada ---------------------------------
elegibles = por_tipo[(por_tipo["n_d"] >= 100) & por_tipo["conf_d"].isin(["alta", "tolerable"])]
extremo = elegibles.loc[elegibles["d"].idxmin(), "BPCOD"]
print(f"\nTipo de d más bajo entre los elegibles: {extremo} — {ETIQUETAS[extremo]}")

# --- 3. Contraste contra el resto, con la covarianza del diseño ------------------
def z_lineal(df, dominio, num, den, peso="FAC_DEL"):
    """Razón del dominio y su variable linealizada (cero fuera del dominio)."""
    w = df[peso].to_numpy(); m = dominio.to_numpy()
    y = (df[num].to_numpy() & m).astype(float)
    x = (df[den].to_numpy() & m).astype(float)
    X = (w * x).sum(); r = (w * y).sum() / X
    return r, w * (y - r * x) / X

def var_diseno(df, z):
    """Varianza de diseño de un total linealizado (misma fórmula que razon_taylor)."""
    tot = (pd.DataFrame({"estrato": df["estrato"].to_numpy(),
                         "upm": df["upm_diseno"].to_numpy(), "z": z})
           .groupby(["estrato", "upm"], sort=False)["z"].sum().reset_index())
    g = tot.groupby("estrato")["z"]
    n_h = g.transform("size")
    dev2 = (tot["z"] - g.transform("mean")) ** 2
    ok = n_h > 1
    return (dev2[ok] * n_h[ok] / (n_h[ok] - 1)).sum()

filas_c = []
for cod in ["01", extremo]:
    dom_a = UNIVERSO & (inc["BPCOD"] == cod)
    dom_b = UNIVERSO & (inc["BPCOD"] != cod)
    for p, (num, den) in DEFINICION.items():
        ra, za = z_lineal(inc, dom_a, num, den)
        rb, zb = z_lineal(inc, dom_b, num, den)
        ee = np.sqrt(var_diseno(inc, za - zb))      # EE de la diferencia
        filas_c.append({"BPCOD": cod, "tipo": ETIQUETAS[cod], "param": p,
                        "tipo_pct": 100 * ra, "resto_pct": 100 * rb,
                        "dif_pp": 100 * (ra - rb), "ee_dif_pp": 100 * ee,
                        "z": (ra - rb) / ee, "razon_vs_resto": ra / rb})
contraste = pd.DataFrame(filas_c)
print("\nContraste contra el resto de los delitos (2020–2024):\n")
print(contraste.round(2).to_string(index=False))

# --- 4. Estabilidad año por año de los dos extremos -----------------------------
filas_a = []
for cod in ["01", extremo]:
    for a, g in inc.groupby("anio_ref"):
        dom = (UNIVERSO & (inc["BPCOD"] == cod)).loc[g.index]
        filas_a.append({"BPCOD": cod, "anio_ref": a, **estimar(g, dom)})
por_anio = pd.DataFrame(filas_a)
print("\nPor año (%):\n")
print(por_anio.assign(**{c: 100 * por_anio[c] for c in ["d", "ee_d", "k", "ee_k", "R", "ee_R"]})
      [["BPCOD", "anio_ref", "n_d", "d", "ee_d", "n_k", "k", "ee_k", "R", "ee_R"]]
      .round(2).to_string(index=False))

**Lo que leo en el control positivo.**

1. **La medición queda validada.** El robo de vehículo tiene d = 74.7% (IC de 70 a 79%), 7.4 veces la del resto de los delitos (z = 27), y se mantiene entre 68% y 83% los cinco años. Si los denominadores estuvieran mal construidos o el estimador estuviera aplastado hacia la media, esto no ocurriría. La regla fijada eligió como extremo opuesto el hostigamiento o abuso sexual, con d = 3.3%: entre los dos extremos hay una razón de 23 a 1.
2. **Lo que no se puede leer del hostigamiento.** Su k de 83.7% descansa en 36 denuncias en cinco años, entre 1 y 11 por año, así que no la interpreto. Su d anual brinca de 0.5% a 7.2%; solo reporto la agrupada.
3. **Un hallazgo sobre k.** En robo de vehículo, k = 84%, casi 20 puntos arriba del resto (z = 6.7). Cuando la víctima necesita la carpeta porque la aseguradora la exige, la barrera institucional se reduce mucho. Esto sugiere que k depende de quién exige la carpeta y del tipo de delito más que del territorio. Aun así, 16% de esas denuncias no llegan a carpeta, en el delito con más incentivo para que lleguen. Lo presento como hipótesis: con estos datos no puedo separar el efecto del seguro de otras diferencias del tipo de delito.
4. **El robo de vehículo no carga solo la conclusión.** Sin él, entre los tipos con confiabilidad alta, d todavía va de 3.3% a 25.7%, unas 8 veces. La comparación formal con las alcaldías la hago más adelante con el mismo tratamiento en los dos lados: componente de varianza contra componente de varianza.

### 4.2 ¿Los extremos son parejos entre alcaldías?

In [ ]:
# ¿Los extremos son parejos entre alcaldías? + control_positivo_tipos.csv

# Para cada extremo: d, k y R directos por alcaldía de ocurrencia, con los 5
# años agrupados (por año la muestra no alcanza: ~6 robos de vehículo por
# alcaldía).
#
# PRUEBA DE HOMOGENEIDAD (Q de Cochran), en escala θ = arcsen√p
#   Q = Σ_a w_a (θ_a − θ̄)²,  w_a = 1 / Var(θ_a),  θ̄ = media ponderada
#   Var(θ_a) ≈ Var(p_a) / (4 p_a (1 − p_a))   (método delta)
#   Bajo homogeneidad, Q ~ χ² con (alcaldías − 1) gl.
#   I² = (Q − gl)/Q: fracción de la dispersión que excede al ruido muestral.
#
# Se excluyen alcaldías con n < 10 o con estimación en 0 o en 1 (degeneradas,
# EE = 0, peso infinito). Si quedan menos de 3, no se prueba.
#
# CÓMO LEERLA: con ~28 casos por alcaldía, la prueba tiene POCA potencia.
# "No rechaza" significa "compatible con parejo", no "demostrado parejo".
# Lo informativo es que TODAS las alcaldías queden en el rango alto, que es lo
# que la hipótesis del seguro predice.

ALC16 = sorted(datos["maestro"]["cve_mun"].unique())
NOMBRES = datos["maestro"].drop_duplicates("cve_mun").set_index("cve_mun")["nombre"]

filas_alc = []
for cod in ["01", extremo]:
    for a in ALC16:
        dom = UNIVERSO & (inc["BPCOD"] == cod) & (inc["alc_ocu"] == a)
        if dom.sum() == 0:
            continue
        # k queda indefinida (0/0) donde no hubo denuncias: se registra como NaN
        with np.errstate(invalid="ignore", divide="ignore"):
            filas_alc.append({"BPCOD": cod, "alc": a, "nombre": NOMBRES[a], **estimar(inc, dom)})
por_alc = pd.DataFrame(filas_alc)

def homogeneidad(g, p):
    est, ee, n = g[p].to_numpy(), g[f"ee_{p}"].to_numpy(), g[f"n_{p}"].to_numpy()
    ok = (est > 0) & (est < 1) & (ee > 0) & (n >= 10)
    if ok.sum() < 3:
        return {"alcaldias_usadas": int(ok.sum())}
    th = np.arcsin(np.sqrt(est[ok]))
    v = ee[ok] ** 2 / (4 * est[ok] * (1 - est[ok]))
    w = 1 / v
    th_bar = (w * th).sum() / w.sum()
    Q = (w * (th - th_bar) ** 2).sum()
    gl = ok.sum() - 1
    return {"alcaldias_usadas": int(ok.sum()), "Q": Q, "gl": gl, "p_valor": chi2.sf(Q, gl),
            "I2": max(0.0, (Q - gl) / Q) if Q > 0 else 0.0,
            "p_min_pct": 100 * est[ok].min(), "p_max_pct": 100 * est[ok].max(),
            "nivel_comun_pct": 100 * np.sin(th_bar) ** 2}

filas_h = []
for cod, g in por_alc.groupby("BPCOD"):
    for p in ["d", "k", "R"]:
        filas_h.append({"BPCOD": cod, "tipo": ETIQUETAS[cod], "param": p, **homogeneidad(g, p)})
homog = pd.DataFrame(filas_h)

v = por_alc.query("BPCOD == '01'")[["nombre", "n_d", "d", "ee_d", "n_k", "k", "ee_k"]].copy()
for c in ["d", "ee_d", "k", "ee_k"]:
    v[c] = 100 * v[c]
print("Robo total de vehículo por alcaldía de ocurrencia, 2020–2024 (%):\n")
print(v.sort_values("d", ascending=False).round(1).to_string(index=False))
print("\nHomogeneidad entre alcaldías (Q de Cochran en escala θ):\n")
print(homog.round(3).to_string(index=False))

# --- Entregable: todo el control positivo en un CSV, por bloques ----------------
ENTREGABLE = pd.concat([
    por_tipo.assign(bloque="tipo_ciudad_2020_2024"),
    contraste.assign(bloque="contraste_vs_resto"),
    por_anio.assign(bloque="extremos_por_anio", tipo=por_anio["BPCOD"].map(ETIQUETAS)),
    por_alc.assign(bloque="extremos_por_alcaldia", tipo=por_alc["BPCOD"].map(ETIQUETAS)),
    homog.assign(bloque="homogeneidad_entre_alcaldias"),
], ignore_index=True)
ENTREGABLE.insert(0, "bloque", ENTREGABLE.pop("bloque"))
ENTREGABLE.to_csv(RUTA_SALIDA / "control_positivo_tipos.csv", index=False, encoding="utf-8-sig")
print(f"\nGuardado: {RUTA_SALIDA / 'control_positivo_tipos.csv'}  ({len(ENTREGABLE)} filas)")

**Lo que leo por alcaldía.** El robo de vehículo sale alto y parejo: las 16 alcaldías quedan entre 65% y 90% en d, y la prueba de homogeneidad no rechaza (p = 0.52, I² = 0). Con unos 28 casos por alcaldía la prueba tiene poca potencia, así que lo que pesa es que ninguna alcaldía cae fuera del rango alto, como predice la hipótesis del seguro. El hostigamiento o abuso sexual sale bajo y parejo, entre 1% y 12% (p = 0.66).

El p = 0.022 de R en robo de vehículo no lo trato como hallazgo: es una de seis pruebas, y con corrección de Bonferroni el umbral es 0.008. Viene sobre todo de Cuauhtémoc (97% en k) y de alcaldías con k = 100% en muestras de 4 a 14 denuncias.

Dentro de un mismo tipo de delito, las alcaldías no se distinguen entre sí. Entre tipos, la diferencia es enorme.

## 5. Heterogeneidad por tipo de delito

Si la variación no es territorial, la busco donde sí puede estar: en el tipo de delito. Estimo los tres parámetros por tipo a nivel ciudad y después comparo, con el mismo estimador, cuánto varían entre tipos y cuánto entre alcaldías.

### 5.1 d, k y R por tipo de delito

In [ ]:
# d, k y R por tipo de delito, ciudad, agrupado y por año

# INTERVALOS EN ESCALA LOGIT
#   El intervalo de Wald (p ± 1.96·EE) se sale de [0, 1] o queda simétrico
#   donde no debe, justo en los extremos que más me interesan (hostigamiento
#   con d = 3%, vehículo con k = 84%). Por eso se construye en logit y se
#   regresa:
#     ℓ = log(p / (1 − p)),   EE(ℓ) ≈ EE(p) / (p (1 − p))   (método delta)
#     IC = expit(ℓ ± 1.96·EE(ℓ))
#   Queda dentro de [0, 1] y es asimétrico cerca de los bordes, como debe ser.
#   Para estimaciones degeneradas (0 o 1) no se reporta intervalo.
#
# PERIODOS: "2020_2024" (cinco levantamientos agrupados, lo principal) y cada
# año por separado (para la aplicación y para ver estabilidad).
#
# "publicable" = confiabilidad alta o tolerable. La aplicación debe mostrar el
# resto con advertencia o no mostrarlo.
#
# incidentes_pond: total de incidentes del tipo, expandido con FAC_DEL. Sirve
# para ponderar y para que la aplicación muestre cuánto pesa cada tipo.

def ic_logit(est, ee, nivel=0.95):
    """Intervalo de confianza construido en logit y regresado a proporción."""
    z = norm.ppf(0.5 + nivel / 2)
    est, ee = np.asarray(est, float), np.asarray(ee, float)
    ok = (est > 0) & (est < 1) & (ee > 0)
    lo, hi = np.full(est.shape, np.nan), np.full(est.shape, np.nan)
    lg = np.log(est[ok] / (1 - est[ok]))
    se_lg = ee[ok] / (est[ok] * (1 - est[ok]))
    lo[ok] = 1 / (1 + np.exp(-(lg - z * se_lg)))
    hi[ok] = 1 / (1 + np.exp(-(lg + z * se_lg)))
    return lo, hi

periodos = [("2020_2024", inc.index)] + [(str(a), g.index) for a, g in inc.groupby("anio_ref")]
filas = []
for periodo, idx in periodos:
    sub = inc.loc[idx]
    for cod in sorted(inc.loc[UNIVERSO, "BPCOD"].unique()):
        dom = (UNIVERSO & (inc["BPCOD"] == cod)).loc[idx]
        with np.errstate(invalid="ignore", divide="ignore"):
            r = estimar(sub, dom)
        inc_pond = sub.loc[dom, "FAC_DEL"].sum()
        for p in ["d", "k", "R"]:
            filas.append({"periodo": periodo, "BPCOD": cod, "tipo": ETIQUETAS[cod], "param": p,
                          "est": r[p], "ee": r[f"ee_{p}"], "n": r[f"n_{p}"],
                          "incidentes_pond": inc_pond})

R_tipo = pd.DataFrame(filas)
R_tipo["cv"] = R_tipo["ee"] / R_tipo["est"]
R_tipo["confiabilidad"] = [confiabilidad(e, s, n)
                           for e, s, n in zip(R_tipo["est"], R_tipo["ee"], R_tipo["n"])]
R_tipo["ic95_inf"], R_tipo["ic95_sup"] = ic_logit(R_tipo["est"], R_tipo["ee"])
R_tipo["publicable"] = R_tipo["confiabilidad"].isin(["alta", "tolerable"])

# --- Vista principal: 2020–2024 agrupado, en % ---------------------------------
ag = R_tipo.query("periodo == '2020_2024'").copy()
for c in ["est", "ic95_inf", "ic95_sup"]:
    ag[c] = 100 * ag[c]
ag["valor"] = (ag["est"].round(1).astype(str) + "  [" + ag["ic95_inf"].round(1).astype(str)
               + ", " + ag["ic95_sup"].round(1).astype(str) + "]"
               + np.where(ag["publicable"], "", "  *"))
vista = ag.pivot(index=["BPCOD", "tipo"], columns="param", values="valor")[["d", "k", "R"]]
orden = ag.query("param == 'd'").set_index(["BPCOD", "tipo"])["est"].sort_values(ascending=False).index
print("d, k y R por tipo de delito, CDMX 2020–2024, % [IC 95% logit]   (* = no publicable)\n")
print(vista.loc[orden].to_string())

print("\nCeldas publicables por periodo (de 42 = 14 tipos × 3 parámetros):")
print(R_tipo.groupby("periodo")["publicable"].sum().to_string())

# --- Entregable ------------------------------------------------------------------
R_tipo.to_parquet(RUTA_SALIDA / "R_por_tipo_delito.parquet", index=False)
print(f"\nGuardado: {RUTA_SALIDA / 'R_por_tipo_delito.parquet'}  ({len(R_tipo)} filas)")

**Lo que leo en la tabla por tipo.**

1. **El rango de d es enorme incluso sin el robo de vehículo.** Entre los tipos publicables, d va de 3.3% (hostigamiento o abuso sexual) a 25.7% (robo a casa habitación), con intervalos que no se tocan: una víctima de robo a casa habitación tiene una probabilidad de denunciar casi ocho veces mayor que una víctima de hostigamiento.
2. **En k aparece un patrón que no puedo afirmar.** Los cuatro tipos de más volumen (autopartes, robo en calle, extorsión y fraude al consumidor) tienen k entre 57% y 60%, mientras que robo de vehículo, otro tipo de robo y lesiones están entre 72% y 84%. Si se sostuviera, la barrera institucional sería más alta justo en los delitos más frecuentes, pero los intervalos se enciman bastante (52 a 67 contra 62 a 80).
3. **Violación sexual y secuestro no se pueden estimar con la ENVIPE en la CDMX.** Hay 38 y 17 casos en cinco años. El modelo de supervivencia muestra que la violación se comporta distinto en el registro, pero del lado de la encuesta no hay muestra para medir su d. Lo declaro como limitación.

### 5.2 ¿Varía más entre tipos de delito o entre alcaldías?

In [ ]:
# ¿Varía más la visibilidad entre tipos de delito o entre alcaldías?

# LA COMPARACIÓN JUSTA
#   Mismo estimador en los dos lados; solo cambia la partición de los datos:
#     - Datos: directos 2020–2024 agrupados, FAC_DEL, ocurrencia CDMX,
#       con su varianza de diseño (linealización de Taylor)
#     - Partición A: tipos de delito (solo celdas publicables)
#     - Partición B: las 16 alcaldías (todos los tipos juntos)
#   Para cada partición, modelo de efectos aleatorios en θ = arcsen√p:
#       θ_g = μ + b_g + e_g,   b_g ~ N(0, τ²),   e_g ~ N(0, v_g)
#   τ² es la varianza de la SEÑAL entre categorías, ya descontado el ruido
#   muestral v_g. Esa es la cantidad comparable; los rangos directos no lo son,
#   porque el ruido infla más a las categorías chicas.
#
# ESTIMACIÓN: τ² por REML; IC al 95% por verosimilitud perfil (caída de
# χ²₁(0.95)/2 = 1.92), el mismo criterio de mi cota por verosimilitud perfil.
#
# TRADUCCIONES
#   de_pp: desviación estándar entre categorías en puntos porcentuales,
#          ≈ sen(2μ)·τ (método delta alrededor del nivel común)
#   razon_mas_menos_1de: p(μ + τ) / p(μ − τ), la razón entre una categoría
#          una DE arriba y otra una DE abajo
#
# LA CIFRA CONSERVADORA para el escrito: el LÍMITE INFERIOR de la DE entre
# tipos dividido entre el LÍMITE SUPERIOR de la DE entre alcaldías. Si aun en
# el peor caso es > 1, la afirmación "varía más entre tipos que entre
# alcaldías" se sostiene con 95% de confianza en cada lado.
#
# VERIFICACIÓN CRUZADA: la τ² de alcaldías en d debe quedar del orden de la
# σ²_alc persistente del motor de dos componentes (0.00013): son la misma pregunta con otra
# ruta (media quinquenal directa contra modelo de celdas).
#
# SENSIBILIDAD: tipos sin el robo de vehículo, para que la conclusión no
# dependa de un solo tipo extremo.
def reml_tau2(th, v):
    """REML de τ² en θ_g = μ + b_g + e_g. Devuelve τ², μ y la función ℓ(τ²)."""
    def ll(t2):
        w = 1 / (v + t2)
        mu = (w * th).sum() / w.sum()
        return -0.5 * (np.log(v + t2).sum() + np.log(w.sum()) + (w * (th - mu) ** 2).sum())
    r = minimize_scalar(lambda t: -ll(t), bounds=(0, 2.0), method="bounded",
                        options={"xatol": 1e-12})
    t2 = r.x if ll(r.x) >= ll(0.0) else 0.0          # revisar la frontera
    w = 1 / (v + t2)
    return t2, (w * th).sum() / w.sum(), ll

def ic_perfil(ll, t2_hat, caida=chi2.ppf(0.95, 1) / 2):
    """IC de τ²: valores cuya ℓ cae menos de 'caida' bajo el máximo."""
    lmax = ll(t2_hat)
    malla = np.concatenate([[0.0], np.geomspace(1e-7, 2.0, 4000)])
    dentro = np.array([ll(t) >= lmax - caida for t in malla])
    return malla[dentro].min(), malla[dentro].max()

def resumen_dispersion(est, ee, etiqueta, p):
    est, ee = np.asarray(est), np.asarray(ee)
    th = np.arcsin(np.sqrt(est))
    v = ee ** 2 / (4 * est * (1 - est))              # varianza en θ (delta)
    t2, mu, ll = reml_tau2(th, v)
    lo, hi = ic_perfil(ll, t2)
    a_razon = lambda t: (np.sin(min(mu + np.sqrt(t), np.pi / 2)) ** 2 /
                         np.sin(max(mu - np.sqrt(t), 1e-9)) ** 2)
    return {"param": p, "particion": etiqueta, "categorias": len(est),
            "nivel_comun_pct": 100 * np.sin(mu) ** 2,
            "tau2": t2, "tau2_ic_inf": lo, "tau2_ic_sup": hi,
            "de_pp": 100 * np.sin(2 * mu) * np.sqrt(t2),
            "de_pp_ic_inf": 100 * np.sin(2 * mu) * np.sqrt(lo),
            "de_pp_ic_sup": 100 * np.sin(2 * mu) * np.sqrt(hi),
            "razon_mas_menos_1de": a_razon(t2),
            "rango_directo_pct": 100 * (est.max() - est.min())}

# --- Partición B: las 16 alcaldías, todos los tipos, 2020–2024 ------------------
por_alc_todo = pd.DataFrame([{"alc": a, **estimar(inc, UNIVERSO & (inc["alc_ocu"] == a))}
                             for a in ALC16])

# --- Las tres particiones, para d, k y R ----------------------------------------
ag = R_tipo.query("periodo == '2020_2024'")
filas_disp = []
for p in ["d", "k", "R"]:
    t = ag[(ag["param"] == p) & ag["publicable"]]
    filas_disp.append(resumen_dispersion(t["est"], t["ee"], "tipos", p))
    t_sv = t[t["BPCOD"] != "01"]
    filas_disp.append(resumen_dispersion(t_sv["est"], t_sv["ee"], "tipos_sin_vehiculo", p))
    filas_disp.append(resumen_dispersion(por_alc_todo[p], por_alc_todo[f"ee_{p}"], "alcaldias", p))
dispersion = pd.DataFrame(filas_disp)

print("Dispersión de la señal entre categorías (REML, IC 95% por perfil):\n")
print(dispersion.drop(columns=["tau2", "tau2_ic_inf", "tau2_ic_sup"]).round(2).to_string(index=False))

# --- Cuántas veces más dispersión entre tipos que entre alcaldías ----------------
comp = []
for p in ["d", "k", "R"]:
    g = dispersion[dispersion["param"] == p].set_index("particion")
    alc, alc_sup = g.loc["alcaldias", "tau2"], g.loc["alcaldias", "tau2_ic_sup"]
    for tp in ["tipos", "tipos_sin_vehiculo"]:
        comp.append({"param": p, "tipos": tp,
                     "veces_puntual": np.sqrt(g.loc[tp, "tau2"] / alc) if alc > 0 else np.inf,
                     "veces_conservador": np.sqrt(g.loc[tp, "tau2_ic_inf"] / alc_sup)})
comparacion = pd.DataFrame(comp)
print("\nDE entre tipos ÷ DE entre alcaldías:")
print("  puntual      = estimación ÷ estimación")
print("  conservador  = límite inferior (tipos) ÷ límite superior (alcaldías)\n")
print(comparacion.round(2).to_string(index=False))

print(f"\nVerificación cruzada, τ² de alcaldías en d: "
      f"{dispersion.query('param == \"d\" and particion == \"alcaldias\"')['tau2'].iloc[0]:.6f} "
      f"(σ²_alc persistente, motor de dos componentes: {verif.set_index('param').loc['d', 's2_alc']:.6f})")

# Guardo la comparación junto a las estimaciones por tipo
dispersion.to_csv(RUTA_SALIDA / "dispersion_tipos_vs_alcaldias.csv", index=False, encoding="utf-8-sig")

**Lo que leo en la comparación.**

- **En d y R la conclusión se sostiene incluso en el peor caso.** Sin el robo de vehículo, la desviación estándar de la señal entre tipos es de 7.7 puntos en d y 5.6 en R; entre alcaldías, de 0.6 y 0.5. Con los límites desfavorables (el inferior para tipos y el superior para alcaldías), la variación entre tipos es al menos 2.75 veces la territorial en d y 2.25 veces en R; con el robo de vehículo, 5.5 y 4.9 veces. La visibilidad del delito depende mucho más de qué le pasó a la víctima que de dónde ocurrió.
- **En k no puedo afirmarlo.** El cociente conservador queda en 0.95 con el robo de vehículo y en 0 sin él. Con estos datos no distingo si la barrera institucional varía más entre tipos que entre alcaldías, lo que es coherente con que k se estima solo sobre los incidentes denunciados, el 11% del total.
- **Verificación cruzada.** La τ² territorial de d por esta ruta (0.000105) coincide en orden de magnitud con la σ²_alc persistente del motor de dos componentes (0.000132): son dos caminos distintos a la misma cantidad.
- **Composición.** Las diferencias entre alcaldías incluyen su mezcla de delitos: una alcaldía con más robo de vehículo tiene más d solo por eso. La variación territorial que mido ya contiene ese componente, así que el efecto puramente territorial sería, si acaso, todavía menor.

## 6. Figura: plano d–k y tasa de carpetas

**Decisiones de diseño.**

- **Directos agrupados 2020–2024, no modelados.** Si dibujara las estimaciones modeladas, las alcaldías se encimarían por construcción, porque el Fay-Herriot las encoge hacia la media cuando σ²_u es chica. Con los directos, el encimamiento lo producen los datos y no el estimador, y la figura vale como evidencia.
- **Elipses con la covarianza de diseño entre d̂ y k̂.** Las dos salen de la misma muestra, así que calculo su covarianza en lugar de suponerla cero.
- **El color es la tasa de carpetas, el mismo en los dos paneles.** Si R explicara las carpetas, los colores se ordenarían en el plano.
- **El panel b está en razón respecto a la ciudad y en escala logarítmica,** para que R (encuesta) y la tasa de carpetas (registro) compartan eje.

In [ ]:
# Figura: plano d–k con regiones de confianza + R contra carpetas

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import spearmanr

def cov_diseno(df, z1, z2):
    """Covarianza de diseño entre dos totales linealizados (misma fórmula que
    var_diseno, con el producto cruzado en lugar del cuadrado)."""
    tot = (pd.DataFrame({"estrato": df["estrato"].to_numpy(), "upm": df["upm_diseno"].to_numpy(),
                         "z1": z1, "z2": z2})
           .groupby(["estrato", "upm"], sort=False)[["z1", "z2"]].sum().reset_index())
    g = tot.groupby("estrato")
    n_h = g["z1"].transform("size")
    d1 = tot["z1"] - g["z1"].transform("mean")
    d2 = tot["z2"] - g["z2"].transform("mean")
    ok = n_h > 1
    return (d1[ok] * d2[ok] * n_h[ok] / (n_h[ok] - 1)).sum()

# --- 1. Directos 2020–2024 por alcaldía, con varianzas y covarianza d–k ---------
filas = []
for a in ALC16:
    dom = UNIVERSO & (inc["alc_ocu"] == a)
    rd, zd = z_lineal(inc, dom, "denuncio", "todos")
    rk, zk = z_lineal(inc, dom, "carpeta", "denuncio")
    rR, zR = z_lineal(inc, dom, "carpeta", "todos")
    filas.append({"alc": a, "nombre": NOMBRES[a], "d": rd, "k": rk, "R": rR,
                  "var_d": var_diseno(inc, zd), "var_k": var_diseno(inc, zk),
                  "var_R": var_diseno(inc, zR), "cov_dk": cov_diseno(inc, zd, zk)})
plano = pd.DataFrame(filas)
plano["corr_dk"] = plano["cov_dk"] / np.sqrt(plano["var_d"] * plano["var_k"])

# --- 2. Tasa de carpetas 2020–2024, con completitud 2024 -------------------------
m = datos["maestro"][["cve_mun", "anio_ref", "pob_18_conapo"]].merge(
    datos["completitud"][["cve_mun", "anio_ref", "C_nucleo_corr"]],
    on=["cve_mun", "anio_ref"], validate="1:1")
C5 = m.groupby("cve_mun").agg(C=("C_nucleo_corr", "sum"), pob=("pob_18_conapo", "sum"))
C5["tasa_C"] = 1e5 * C5["C"] / C5["pob"]
plano = plano.merge(C5["tasa_C"], left_on="alc", right_index=True, validate="1:1")
TASA_C_CIUDAD = 1e5 * C5["C"].sum() / C5["pob"].sum()
ciudad5 = estimar(inc, UNIVERSO)

print(plano[["nombre", "d", "k", "R", "corr_dk", "tasa_C"]].round(3).to_string(index=False))
print(f"\nRazón máx/mín: tasa de carpetas {plano['tasa_C'].max() / plano['tasa_C'].min():.2f}, "
      f"R directo {plano['R'].max() / plano['R'].min():.2f}")
rho, pv = spearmanr(plano["R"], plano["tasa_C"])
print(f"Spearman R directo vs tasa de carpetas: {rho:.2f} (p = {pv:.2f})")

# --- 3. La figura -----------------------------------------------------------------
ABREV = {"Azcapotzalco": "AZC", "Coyoacán": "COY", "Cuajimalpa de Morelos": "CUJ",
         "Gustavo A. Madero": "GAM", "Iztacalco": "IZC", "Iztapalapa": "IZP",
         "La Magdalena Contreras": "MAC", "Milpa Alta": "MIL", "Álvaro Obregón": "AOB",
         "Tláhuac": "TLH", "Tlalpan": "TLP", "Xochimilco": "XOC", "Benito Juárez": "BJU",
         "Cuauhtémoc": "CUH", "Miguel Hidalgo": "MHI", "Venustiano Carranza": "VCA"}

mpl.rcParams.update({"font.family": "serif", "font.size": 8})
norm_c = mpl.colors.LogNorm(plano["tasa_C"].min(), plano["tasa_C"].max())
cmap = mpl.colormaps["viridis"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.16, 3.4),          # ancho de doble columna IEEE
                               gridspec_kw={"width_ratios": [1.25, 1]})

# Desplazamiento manual (en puntos) de las etiquetas del centro, que se enciman.
# Las que se mueven llevan una línea guía hacia su punto.
DESPLAZA = {"CUH": (16, 12), "MIL": (-24, 7), "AOB": (-28, -1), "XOC": (14, -5),
            "IZP": (-20, -8), "GAM": (14, 3), "CUJ": (12, -9), "TLH": (-14, -4),
            "MHI": (-16, 2), "VCA": (-10, 8), "BJU": (4, 9), "IZC": (12, 4)}

# (a) Elipse de confianza 95%: contorno de la normal bivariada con radio √χ²₂(0.95).
#     Sus ejes son los vectores propios de la matriz de covarianza; sus semiejes,
#     radio × √(valor propio).
R95 = np.sqrt(chi2.ppf(0.95, 2))
for _, r in plano.iterrows():
    S = np.array([[r.var_d, r.cov_dk], [r.cov_dk, r.var_k]]) * 1e4      # a puntos²
    val, vec = np.linalg.eigh(S)
    ang = np.degrees(np.arctan2(vec[1, 1], vec[0, 1]))
    col = cmap(norm_c(r.tasa_C))
    ax1.add_patch(Ellipse((100 * r.d, 100 * r.k), 2 * R95 * np.sqrt(val[1]), 2 * R95 * np.sqrt(val[0]),
                          angle=ang, facecolor=col, alpha=0.08, edgecolor=col, lw=0.8))
    ax1.plot(100 * r.d, 100 * r.k, "o", ms=3, color=col)
    dx, dy = DESPLAZA.get(ABREV[r.nombre], (2, 2))
    movida = (dx, dy) != (2, 2)
    ax1.annotate(ABREV[r.nombre], (100 * r.d, 100 * r.k), xytext=(dx, dy), textcoords="offset points",
                 fontsize=6, ha="right" if dx < 0 else "left", va="center" if movida else "bottom",
                 arrowprops=dict(arrowstyle="-", lw=0.4, color="0.35", shrinkA=0, shrinkB=1.5) if movida else None)
ax1.axvline(100 * ciudad5["d"], color="0.4", ls=":", lw=0.8)
ax1.axhline(100 * ciudad5["k"], color="0.4", ls=":", lw=0.8)
ax1.set_xlabel("d: % de delitos denunciados")
ax1.set_ylabel("k: % de denuncias que llegan a carpeta")
ax1.set_title("(a) d y k por alcaldía, región de confianza 95%", fontsize=8)
ax1.autoscale_view()

# (b) Razón respecto a la ciudad, ordenado por tasa de carpetas
y_R  = plano["R"] / ciudad5["R"]
lo_R = (plano["R"] - 1.96 * np.sqrt(plano["var_R"])) / ciudad5["R"]
hi_R = (plano["R"] + 1.96 * np.sqrt(plano["var_R"])) / ciudad5["R"]
y_C  = plano["tasa_C"] / TASA_C_CIUDAD
orden = np.argsort(y_C.to_numpy())
for i, j in enumerate(orden):
    col = cmap(norm_c(plano["tasa_C"].iloc[j]))
    ax2.plot([lo_R.iloc[j], hi_R.iloc[j]], [i, i], color="0.55", lw=1)
    ax2.plot(y_R.iloc[j], i, "o", ms=3.5, mfc="white", mec="0.25")
    ax2.plot(y_C.iloc[j], i, "s", ms=4, color=col)
ax2.set_xscale("log")
ax2.set_xticks([0.4, 0.5, 0.7, 1, 1.5, 2, 3])
ax2.set_xticklabels(["0.4", "0.5", "0.7", "1", "1.5", "2", "3"])
ax2.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax2.axvline(1, color="0.4", ls=":", lw=0.8)
ax2.set_yticks(range(16))
ax2.set_yticklabels([ABREV[plano["nombre"].iloc[j]] for j in orden], fontsize=6)
ax2.set_xlabel("Valor de la alcaldía ÷ valor de la ciudad (escala log)")
ax2.set_title("(b) R (○, IC 95%) y tasa de carpetas (■)", fontsize=8)

sm = mpl.cm.ScalarMappable(norm=norm_c, cmap=cmap)
cb = fig.colorbar(sm, ax=[ax1, ax2], fraction=0.025, pad=0.02)
cb.set_label("Carpetas por 100 mil hab. 18+", fontsize=7)
cb.set_ticks([450, 600, 800, 1000, 1500]); cb.set_ticklabels(["450", "600", "800", "1000", "1500"])
cb.ax.minorticks_off()

for ext in ["png", "pdf"]:            # PDF vectorial para el escrito IEEE
    fig.savefig(RUTA_SALIDA / f"fig_plano_dk_vs_carpetas.{ext}", dpi=300, bbox_inches="tight")
plt.show()

plano.to_parquet(RUTA_SALIDA / "plano_dk_directos_2020_2024.parquet", index=False)

**Lo que leo en la figura.**

- El orden de R no tiene relación con el de las carpetas (Spearman 0.06, p = 0.82). Cuauhtémoc, con 2.1 veces la tasa de carpetas de la ciudad, tiene una R justo en el promedio.
- La razón máx/mín de R directa es 1.84, contra 3.52 de las carpetas. Ese 1.84 incluye ruido muestral, porque son estimaciones directas; la señal entre alcaldías es mucho menor (desviación estándar de 0.5 puntos). No lo presento como variación real.

*Pendiente:* desencimar las etiquetas del centro (CUH, AOB, MIL).

## 7. Mapa corregido

**Decisiones.**

- **Qué R uso.** La modelada y ajustada al total de ciudad, que es mi estimación final. En la simulación Monte Carlo trabajo en la escala del modelo, θ ~ N(θ̂, ECM), regreso con sen²θ y aplico el factor del ajuste. Verifico que p_modelo es exactamente la media corregida por sesgo de esa transformación.
- **Factor de completitud.** C_nucleo_corr es C_nucleo × factor_completitud, distinto de 1 solo en 2024. Lo aplico a las cuatro variantes de C bajo el supuesto de que el rezago de registro es el mismo en todas.
- **Solo el orden es interpretable.** C cuenta solo las categorías de cada variante y R se refiere a todos los delitos, así que C/R no estima la incidencia real sino un índice proporcional a ella dentro de cada variante. La diferencia de nivel la analizo en la ruta administrativa.

### 7.1 Corrección con incertidumbre Monte Carlo

In [ ]:
# Mapa corregido: C / R con incertidumbre por Monte Carlo

# LA CORRECCIÓN
#   Si R_a = P(carpeta | delito), los delitos que ocurrieron son V_a = C_a / R_a.
#   Tasa corregida = V_a / población 18+. Si R fuera igual en todas las
#   alcaldías, el orden corregido sería idéntico al observado. Cuánto se
#   mueve el orden es la medida de cuánto deforma R el mapa.
#
# MONTE CARLO (N_SIM simulaciones)
#   1. θ_at ~ N(θ̂_at, ECM_θ) del Fay-Herriot agrupado (el modelo que elegí)
#   2. R_at = sen²(θ_at) × factor de benchmarking del año
#   3. V_at = C_at / R_at  para cada variante de C (con completitud 2024)
#   4. Periodo agrupado: V_a = Σ_t V_at y población Σ_t pob_at, para que cada
#      año pese con sus propias carpetas
#   5. En cada simulación: tasas, rangos y Spearman contra el orden observado
#   Resultado: mediana e IC 95% de la tasa corregida y del RANGO de cada
#   alcaldía, y la probabilidad de que se mueva más de 2 posiciones.
#
# SUPUESTO: simulaciones independientes entre alcaldías y años. El FH
# agrupado trata los efectos de área como independientes; la incertidumbre de
# β induciría una correlación positiva entre celdas que mueve a todas en la
# misma dirección y casi no afecta el ORDEN. Para rangos, el supuesto es
# conservador: exagera un poco cuánto pueden moverse.
#
# R_efectiva: C_a / V_a, la R que implica el periodo (en un año es R_at).

N_SIM = 5000
rng = np.random.default_rng(20261013)
VARIANTES = ["C_nucleo", "C_nucleo_alta", "C_nucleo_vf", "C_amplio"]

# --- 1. R modelada: θ y ECM del FH agrupado + factor de benchmarking -------------
fh = (datos["modelos"].query("metodo == '2_FH_agrupado' and param == 'R'")
      [["alc", "anio_ref", "theta", "ecm_theta"]]
      .merge(datos["estimaciones"].query("param == 'R'")
             [["alc", "anio_ref", "p_modelo", "p_bench_consistente", "factor_consistente"]],
             on=["alc", "anio_ref"], validate="1:1"))
# Verificación: p_modelo = ½ − ½·cos(2θ)·exp(−2·ECM), la media de sen²θ
assert np.allclose(0.5 - 0.5 * np.cos(2 * fh["theta"]) * np.exp(-2 * fh["ecm_theta"]), fh["p_modelo"])

# --- 2. Carpetas (4 variantes) con completitud, población y R en una tabla ------
base_C = (datos["maestro"][["cve_mun", "anio_ref", "pob_18_conapo"] + VARIANTES]
          .merge(datos["completitud"][["cve_mun", "anio_ref", "factor_completitud"]],
                 on=["cve_mun", "anio_ref"], validate="1:1")
          .rename(columns={"cve_mun": "alc"})
          .merge(fh, on=["alc", "anio_ref"], validate="1:1")
          .sort_values(["alc", "anio_ref"]).reset_index(drop=True))
for v in VARIANTES:
    base_C[v + "_corr"] = base_C[v] * base_C["factor_completitud"]

# --- 3. Simulaciones de R: matriz (80 celdas × N_SIM) ---------------------------
theta_sim = (base_C["theta"].to_numpy()[:, None]
             + rng.standard_normal((len(base_C), N_SIM)) * np.sqrt(base_C["ecm_theta"].to_numpy())[:, None])
R_sim = np.sin(theta_sim) ** 2 * base_C["factor_consistente"].to_numpy()[:, None]

# --- 4. Agregación por alcaldía y periodo -----------------------------------------
alcs = sorted(base_C["alc"].unique())
idx_alc = base_C["alc"].map({a: i for i, a in enumerate(alcs)}).to_numpy()
anio_ord = base_C["anio_ref"].to_numpy()
periodos = {str(a): anio_ord == a for a in sorted(set(anio_ord))}
periodos["2020_2024"] = np.ones(len(base_C), bool)

def agrega(mascara, valores):
    """Suma por alcaldía de las celdas seleccionadas (funciona con matrices)."""
    out = np.zeros((len(alcs),) + valores.shape[1:])
    np.add.at(out, idx_alc[mascara], valores[mascara])
    return out

def rangos(x):
    """Rango 1 = tasa más alta, por columna."""
    return (-x).argsort(axis=0).argsort(axis=0) + 1

filas = []
pob = base_C["pob_18_conapo"].to_numpy()
for variante in VARIANTES:
    C = base_C[variante + "_corr"].to_numpy()
    for periodo, mk in periodos.items():
        C_a, pob_a = agrega(mk, C), agrega(mk, pob)
        tasa_obs = 1e5 * C_a / pob_a
        V_sim = agrega(mk, C[:, None] / R_sim)               # delitos implicados
        tasa_corr = 1e5 * V_sim / pob_a[:, None]
        R_ef_sim = C_a[:, None] / V_sim
        rk_obs = rangos(tasa_obs[:, None])[:, 0]
        rk_sim = rangos(tasa_corr)
        mov = np.abs(rk_sim - rk_obs[:, None])
        rho_sim = np.array([spearmanr(tasa_obs, tasa_corr[:, s])[0]
                            for s in range(0, N_SIM, 10)])      # 1 de cada 10: basta
        for i, a in enumerate(alcs):
            filas.append({
                "variante_C": variante, "periodo": periodo, "alc": a, "nombre": NOMBRES[a],
                "C_corr": C_a[i], "pob_18": pob_a[i], "tasa_obs": tasa_obs[i],
                "R_efectiva": np.median(R_ef_sim[i]),
                "R_ef_ic_inf": np.quantile(R_ef_sim[i], 0.025),
                "R_ef_ic_sup": np.quantile(R_ef_sim[i], 0.975),
                "tasa_corr": np.median(tasa_corr[i]),                  # índice: solo el orden es interpretable
                "tasa_corr_ic_inf": np.quantile(tasa_corr[i], 0.025),
                "tasa_corr_ic_sup": np.quantile(tasa_corr[i], 0.975),
                "rango_obs": int(rk_obs[i]),
                "rango_corr_mediana": np.median(rk_sim[i]),
                "rango_corr_ic_inf": np.quantile(rk_sim[i], 0.025),
                "rango_corr_ic_sup": np.quantile(rk_sim[i], 0.975),
                "prob_mueve_mas_2": (mov[i] > 2).mean(),
                "spearman_mediana": np.median(rho_sim),
                "spearman_ic_inf": np.quantile(rho_sim, 0.025),
                "spearman_ic_sup": np.quantile(rho_sim, 0.975)})
mapa = pd.DataFrame(filas)

# --- 5. Resumen: ¿cuánto se mueve el mapa? ----------------------------------------
resumen_mapa = mapa.groupby(["variante_C", "periodo"]).agg(
    spearman=("spearman_mediana", "first"),
    spearman_ic_inf=("spearman_ic_inf", "first"),
    razon_obs=("tasa_obs", lambda s: s.max() / s.min()),
    razon_corr=("tasa_corr", lambda s: s.max() / s.min()),
    alc_prob_mover_mas_2_sup_50=("prob_mueve_mas_2", lambda s: int((s > 0.5).sum())))
print("Orden observado contra corregido, por variante de C y periodo:\n")
print(resumen_mapa.round(3).to_string())

print("\nDetalle, C núcleo, 2020–2024 agrupado:\n")
print(mapa.query("variante_C == 'C_nucleo' and periodo == '2020_2024'").sort_values("rango_obs")
      [["nombre", "tasa_obs", "R_efectiva", "rango_obs", "rango_corr_mediana",
        "rango_corr_ic_inf", "rango_corr_ic_sup", "prob_mueve_mas_2"]].round(3).to_string(index=False))

mapa.to_parquet(RUTA_SALIDA / "mapa_corregido.parquet", index=False)
print(f"\nGuardado: {RUTA_SALIDA / 'mapa_corregido.parquet'}  ({len(mapa)} filas)")

**Lo que leo en el mapa corregido.**

- **La magnitud de la desigualdad sobrevive.** Con C núcleo, la razón máx/mín pasa de 3.52 a 3.41; en las cuatro variantes la corrección apenas la reduce.
- **Los extremos no se mueven.** Cuauhtémoc queda primera y Cuajimalpa última en el 100% de las simulaciones.
- **El orden general se conserva.** Spearman de 0.87 (límite inferior 0.78), y de 0.79 a 0.89 según la variante de C.
- **La zona media se reacomoda.** Coyoacán baja del lugar 8 al 13 (probabilidad de moverse más de dos lugares: 0.89) y Tlalpan del 12 al 15 (0.55); las dos tienen las R más altas (8.3% y 7.9%). Milpa Alta sube del 15 al 11 (0.60), con una de las más bajas (6.1%). En esa zona las tasas de carpetas difieren poco entre sí, y una diferencia chica en R alcanza para cambiar el orden.
- **Los mapas anuales se mueven mucho más.** En 2024 el Spearman baja a 0.61 (límite inferior 0.32) y 9 alcaldías se mueven. La variación de R de un año a otro no es persistente, y corregir cada año con su propia R la traslada al mapa. Por eso uso el mapa agrupado como referencia; cualquier vista anual en la aplicación debe mostrar el intervalo del rango.

**¿Hay circularidad?** El modelo de R usa covariables derivadas de registros administrativos (la tasa SESNSP y el canal digital). Si la parte sintética de R depende de un registro, dividir las carpetas entre R dividiría en parte el registro entre sí mismo. Para descartarlo repito la corrección con la R directa agrupada, que no usa ninguna covariable.

### 7.2 Sensibilidad a la fuente de R y a la definición de carpetas

In [ ]:
# Sensibilidad del mapa corregido a la fuente de R

# TRES FUENTES DE R, solo para el periodo agrupado 2020–2024:
#   FH_benchmark       la principal: modelo con covariables + ajuste
#   directo_2020_2024  directo agrupado, SIN modelo ni covariables (el del plano d–k).
#                      Cierra la objeción de circularidad. Es más ruidoso,
#                      así que se espera que mueva MÁS el orden.
#   fija_ciudad        R igual en todas: orden idéntico por construcción
#                      (Spearman = 1). Es el punto de referencia del resultado
#                      nulo, no una prueba.
#
# Simulación del directo: θ ~ N(arcsen√R̂, Var(R̂)/(4R̂(1−R̂))) con la varianza
# de diseño de los directos agrupados, la misma transformación que en el resto del trabajo.
#
# LECTURA: si los movimientos del mapa corregido (Coyoacán y Tlalpan abajo, Milpa Alta
# arriba) también aparecen con el directo, no son un artefacto de las
# covariables. Si solo aparecen con el FH, sí lo son.
mapa["fuente_R"] = "FH_benchmark"
mk = periodos["2020_2024"]
pob_a = agrega(mk, pob)

# --- Simulaciones del R directo agrupado (orden de alcs) ---------------------------
pl = plano.set_index("alc").loc[alcs]
th_dir = np.arcsin(np.sqrt(pl["R"].to_numpy()))
v_th = pl["var_R"].to_numpy() / (4 * pl["R"].to_numpy() * (1 - pl["R"].to_numpy()))
R_dir_sim = np.sin(th_dir[:, None]
                   + rng.standard_normal((len(alcs), N_SIM)) * np.sqrt(v_th)[:, None]) ** 2

filas = []
for variante in VARIANTES:
    C_a = agrega(mk, base_C[variante + "_corr"].to_numpy())
    tasa_obs = 1e5 * C_a / pob_a
    rk_obs = rangos(tasa_obs[:, None])[:, 0]
    tasa_corr = 1e5 * (C_a[:, None] / R_dir_sim) / pob_a[:, None]
    rk_sim = rangos(tasa_corr)
    mov = np.abs(rk_sim - rk_obs[:, None])
    rho_sim = np.array([spearmanr(tasa_obs, tasa_corr[:, s])[0] for s in range(0, N_SIM, 10)])
    for i, a in enumerate(alcs):
        filas.append({"fuente_R": "directo_2020_2024", "variante_C": variante, "periodo": "2020_2024",
                      "alc": a, "nombre": NOMBRES[a], "C_corr": C_a[i], "pob_18": pob_a[i],
                      "tasa_obs": tasa_obs[i],
                      "R_efectiva": np.median(R_dir_sim[i]),
                      "R_ef_ic_inf": np.quantile(R_dir_sim[i], 0.025),
                      "R_ef_ic_sup": np.quantile(R_dir_sim[i], 0.975),
                      "tasa_corr": np.median(tasa_corr[i]),
                      "tasa_corr_ic_inf": np.quantile(tasa_corr[i], 0.025),
                      "tasa_corr_ic_sup": np.quantile(tasa_corr[i], 0.975),
                      "rango_obs": int(rk_obs[i]), "rango_corr_mediana": np.median(rk_sim[i]),
                      "rango_corr_ic_inf": np.quantile(rk_sim[i], 0.025),
                      "rango_corr_ic_sup": np.quantile(rk_sim[i], 0.975),
                      "prob_mueve_mas_2": (mov[i] > 2).mean(),
                      "spearman_mediana": np.median(rho_sim),
                      "spearman_ic_inf": np.quantile(rho_sim, 0.025),
                      "spearman_ic_sup": np.quantile(rho_sim, 0.975)})
    # R fija: tasa corregida ∝ observada → mismo orden por construcción
    filas += [{"fuente_R": "fija_ciudad", "variante_C": variante, "periodo": "2020_2024",
               "alc": a, "nombre": NOMBRES[a], "tasa_obs": tasa_obs[i], "rango_obs": int(rk_obs[i]),
               "rango_corr_mediana": float(rk_obs[i]), "prob_mueve_mas_2": 0.0,
               "spearman_mediana": 1.0} for i, a in enumerate(alcs)]
mapa = pd.concat([mapa, pd.DataFrame(filas)], ignore_index=True)

# --- Tabla de sensibilidad: 4 variantes de C × 3 fuentes de R ----------------------
sens = (mapa.query("periodo == '2020_2024'")
        .groupby(["variante_C", "fuente_R"])
        .agg(spearman=("spearman_mediana", "first"),
             sp_ic_inf=("spearman_ic_inf", "first"),
             razon_corr=("tasa_corr", lambda s: s.max() / s.min()),
             alc_mueven=("prob_mueve_mas_2", lambda s: int((s > 0.5).sum())))
        .unstack("fuente_R"))
print("Sensibilidad, 2020–2024 agrupado:\n")
print(sens.round(3).to_string())

# --- ¿Los mismos movimientos con las dos fuentes? -----------------------------------
comp = (mapa.query("variante_C == 'C_nucleo' and periodo == '2020_2024' and fuente_R != 'fija_ciudad'")
        .pivot(index=["nombre", "rango_obs"], columns="fuente_R",
               values=["R_efectiva", "rango_corr_mediana"])
        .sort_index(level="rango_obs"))
print("\nC núcleo: R y rango corregido con cada fuente:\n")
print(comp.round(3).to_string())

sel = mapa.query("variante_C == 'C_nucleo' and periodo == '2020_2024'")
r_fh = sel.query("fuente_R == 'FH_benchmark'").set_index("alc")["R_efectiva"]
r_di = sel.query("fuente_R == 'directo_2020_2024'").set_index("alc")["R_efectiva"]
rho, pv = spearmanr(r_fh, r_di.loc[r_fh.index])
print(f"\nSpearman entre el orden de R modelada y R directa: {rho:.2f} (p = {pv:.3f})")

mapa.to_parquet(RUTA_SALIDA / "mapa_corregido.parquet", index=False)
print(f"Guardado de nuevo con las tres fuentes: {len(mapa)} filas")

**Lo que leo en la sensibilidad.**

- **La circularidad queda descartada.** Coyoacán y Tlalpan bajan con las dos fuentes de R: con la modelada quedan en los lugares 13 y 15, y con la directa, sin covariables, en el 14 y el 16. Sus R directas (9.7% y 9.2%) son de hecho las más altas de la ciudad.
- **Lo robusto, en las 12 combinaciones (4 variantes de C × 3 fuentes de R):** la razón corregida va de 2.4 a 4.9, contra 2.5 a 5.0 observada; Cuauhtémoc conserva el primer lugar en todas; el Spearman nunca baja de 0.56.
- **Lo frágil:** con la R directa el orden se mueve más (Spearman de 0.56 a 0.73, límite inferior hasta 0.26), como se espera de un estimador con ruido muestral. Cuajimalpa, última con la modelada, sube al lugar 13 con la directa. El orden de R modelada y el de R directa coinciden solo moderadamente (Spearman 0.60).

Corregir por la probabilidad de registro conserva la magnitud de la desigualdad territorial y su extremo superior, y solo reacomoda posiciones intermedias. Esos movimientos dependen de diferencias en R que la prueba de varianza persistente no distingue del ruido, así que no los presento como hallazgos sobre alcaldías específicas.

### 7.3 Figura del ranking observado contra corregido

Muestro tres columnas (observado, corregido con R modelada y corregido con R directa) para que la robustez se vea en la misma imagen. Destaco en rojo las alcaldías que bajan y en azul las que suben, solo cuando la probabilidad de moverse más de dos lugares supera 0.5. La posición corregida es el orden de la tasa corregida mediana, que no tiene empates. La incertidumbre del rango la reporto en la tabla y no en la figura, porque ahí se volvía ilegible.

In [ ]:
# Figura: ranking observado contra corregido (C núcleo, 2020–2024)

# Tres columnas: observado → corregido con R modelada → corregido con R directa.
# Color: rojo = baja más de 2 lugares con probabilidad > 0.5 (R modelada);
#        azul = sube; gris = el resto.
# Posición corregida = orden de la tasa corregida MEDIANA (sin empates).
# El IC del rango está en mapa_corregido.parquet y va en la tabla del escrito.

sel = mapa.query("variante_C == 'C_nucleo' and periodo == '2020_2024'")
fh_ = sel.query("fuente_R == 'FH_benchmark'").set_index("alc").copy()
di_ = sel.query("fuente_R == 'directo_2020_2024'").set_index("alc").copy()
for src in (fh_, di_):
    src["pos_corr"] = src["tasa_corr"].rank(ascending=False).astype(int)

fig, ax = plt.subplots(figsize=(3.5, 4.2))            # una columna IEEE
X = [0, 1, 2]
for a in alcs:
    y = [fh_.loc[a, "rango_obs"], fh_.loc[a, "pos_corr"], di_.loc[a, "pos_corr"]]
    destaca = fh_.loc[a, "prob_mueve_mas_2"] > 0.5
    col = "#c0392b" if destaca and y[1] > y[0] else ("#2471a3" if destaca else "0.6")
    ax.plot(X, y, "-", color=col, lw=1.4 if destaca else 0.8, alpha=0.9, zorder=2 if destaca else 1)
    ax.plot(X, y, "o", ms=3, color=col, zorder=3)
    etiqueta_col = col if destaca else "0.25"
    ax.text(-0.06, y[0], ABREV[NOMBRES[a]], ha="right", va="center", fontsize=6, color=etiqueta_col)
    ax.text(2.10, y[2], ABREV[NOMBRES[a]], ha="left", va="center", fontsize=6, color=etiqueta_col)

ax.set_xticks(X)
ax.set_xticklabels(["Observado\n(carpetas)", "Corregido\nR modelada", "Corregido\nR directa"], fontsize=7)
ax.set_ylim(16.7, 0.3)
ax.set_yticks([1, 4, 8, 12, 16])
ax.set_ylabel("Rango (1 = tasa más alta)")
ax.set_xlim(-0.45, 2.45)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.set_title("Ranking de alcaldías, C núcleo, 2020–2024", fontsize=8)

for ext in ["png", "pdf"]:
    fig.savefig(RUTA_SALIDA / f"fig_ranking_observado_corregido.{ext}", dpi=300, bbox_inches="tight")
plt.show()

# Tabla que acompaña a la figura en el escrito
tabla_rank = pd.DataFrame({
    "alcaldia": [NOMBRES[a] for a in alcs],
    "rango_obs": fh_.loc[alcs, "rango_obs"].to_numpy(),
    "R_modelada_pct": 100 * fh_.loc[alcs, "R_efectiva"].to_numpy(),
    "rango_corr": fh_.loc[alcs, "pos_corr"].to_numpy(),
    "rango_corr_ic95": [f"{int(i)}–{int(s)}" for i, s in
                        zip(fh_.loc[alcs, "rango_corr_ic_inf"], fh_.loc[alcs, "rango_corr_ic_sup"])],
    "prob_mueve_mas_2": fh_.loc[alcs, "prob_mueve_mas_2"].to_numpy(),
    "rango_corr_R_directa": di_.loc[alcs, "pos_corr"].to_numpy(),
}).sort_values("rango_obs")
print(tabla_rank.round(2).to_string(index=False))
tabla_rank.to_csv(RUTA_SALIDA / "tabla_ranking_corregido.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Verificación: posición de Cuauhtémoc (015) en las 12 combinaciones de C y R
print(mapa.query("periodo == '2020_2024' and alc == '015'")[["variante_C", "fuente_R", "rango_obs", "rango_corr_mediana"]])

## 8. Coherencia interna del modelo

In [ ]:
# Coherencia interna: R modelada frente a d·k modeladas

# En los estimadores directos R̂ = d̂·k̂ por identidad, así que las diferencias
# solo pueden venir de haber modelado y ajustado los tres parámetros por
# separado. Resumo esas diferencias para reportarlas como coherencia interna
# del modelo, no como validación del fenómeno.

cons = datos["consistencia"]
print("Diferencia R − d·k en las 80 celdas alcaldía-año:\n")
print(cons[["dif_R_dk_pp", "dif_rel", "dif_en_ee_R"]].describe().round(3).to_string())
print(f"\nCeldas con |diferencia| > 1 EE de R: {(cons['dif_en_ee_R'].abs() > 1).sum()} de {len(cons)}")
print(f"Máximo |diferencia| en EE de R: {cons['dif_en_ee_R'].abs().max():.2f}")
print("\nMedia de la diferencia por año (pp):")
print(cons.groupby("anio_ref")["dif_R_dk_pp"].mean().round(3).to_string())

En los estimadores directos, $\hat R = \hat d \cdot \hat k$ por identidad: la probabilidad conjunta de denuncia y carpeta es el producto de la marginal por la condicional, y ningún incidente tiene carpeta sin denuncia. La identidad deja de cumplirse solo porque los tres parámetros se modelaron por separado, cada uno con sus covariables y su encogimiento, y se ajustaron por separado a su total de ciudad.

En las 80 celdas alcaldía-año, $\tilde R - \tilde d \cdot \tilde k$ va de −1.76 a +1.15 puntos porcentuales (mediana −0.07). Ninguna diferencia excede 1.45 errores estándar de $\tilde R$, y solo 5 de 80 pasan uno. La media es −0.17 puntos, ligeramente negativa los cinco años, lo que es compatible con que un promedio ponderado de productos no es el producto de los promedios. En términos relativos, las diferencias llegan a −19% y +16% en las celdas con $R$ más baja.

Esta comparación es una prueba de coherencia interna del modelo, no una validación sustantiva. Muestra que modelar por separado no produjo estimaciones incompatibles entre sí; no aporta evidencia sobre si $R$ mide bien la probabilidad de registro. Si una aplicación requiere que la identidad se cumpla exactamente, el ajuste proporcional iterativo la impone por construcción, a costa de que $R$ deje de ser un estimador propio.

## 9. Robustez por geografía de residencia

La geografía principal es la de ocurrencia. Aquí repito las estimaciones asignando cada incidente a la alcaldía donde vive la víctima, para ver si el resultado nulo depende de esa elección.

In [ ]:
# Robustez por geografía de residencia

# RESIDENCIA: d, k y R directos 2020–2024 por alcaldía donde VIVE la víctima
# (residentes de la CDMX, el delito pudo ocurrir en cualquier lugar), contra
# los mismos por alcaldía de OCURRENCIA.
#
# OJO CON EL SPEARMAN: si la señal territorial es casi cero, el orden de las
# alcaldías es casi puro ruido, y dos versiones del mismo ruido (que además
# comparten la mayoría de los incidentes) no tienen por qué coincidir. La
# prueba de robustez relevante no es "¿coincide el orden?", sino "¿el nulo
# territorial se sostiene también por residencia?". Por eso repito la
# dispersión REML con las dos geografías.
#
UNIV_RES = inc["cdmx_res"] & ~inc["excl_vandalismo"]

# --- 1. d, k y R por ocurrencia y por residencia -----------------------------------
filas = []
for a in ALC16:
    r_ocu = estimar(inc, UNIVERSO & (inc["alc_ocu"] == a))
    r_res = estimar(inc, UNIV_RES & (inc["alc_res"] == a))
    filas.append({"alc": a, "nombre": NOMBRES[a],
                  **{f"{p}_ocu": r_ocu[p] for p in "dkR"},
                  **{f"ee_{p}_ocu": r_ocu[f"ee_{p}"] for p in "dkR"},
                  **{f"{p}_res": r_res[p] for p in "dkR"},
                  **{f"ee_{p}_res": r_res[f"ee_{p}"] for p in "dkR"},
                  "n_k_res": r_res["n_k"]})
geo = pd.DataFrame(filas)

print("Spearman ocurrencia vs residencia (directos agrupados):")
for p in "dkR":
    print(f"  {p}: {spearmanr(geo[f'{p}_ocu'], geo[f'{p}_res'])[0]:.2f}")

disp_geo = pd.DataFrame([resumen_dispersion(geo[f"{p}_{g}"], geo[f"ee_{p}_{g}"], g, p)
                         for p in "dkR" for g in ["ocu", "res"]])
print("\nDispersión de la señal entre alcaldías, por geografía (pp, IC 95% perfil):\n")
print(disp_geo[["param", "particion", "nivel_comun_pct", "de_pp", "de_pp_ic_inf", "de_pp_ic_sup"]]
      .round(2).to_string(index=False))

print("\nk por residencia (orden ascendente):\n")
print(geo[["nombre", "k_ocu", "ee_k_ocu", "k_res", "ee_k_res", "n_k_res"]]
      .sort_values("k_res").round(3).to_string(index=False))

Repetí las estimaciones directas agrupadas 2020–2024 asignando cada incidente a la alcaldía de residencia de la víctima en lugar de la de ocurrencia. Las correlaciones de rangos entre ambas geografías son 0.72 en $d$, 0.83 en $k$ y 0.82 en $R$, por encima de las obtenidas con estimaciones anuales (0.55 a 0.89 en $d$) y por debajo del 0.91 que reportan Buil-Gil et al. (2021).

Con una señal territorial cercana a cero, sin embargo, la correlación de rangos no es la prueba adecuada: el orden de las alcaldías es casi todo ruido muestral, y dos versiones de ese ruido no tienen por qué coincidir. La prueba relevante es si el resultado nulo se sostiene con la otra geografía. En $d$ y $R$ se sostiene: la desviación estándar de la señal entre alcaldías por residencia es de 0.9 y 1.1 puntos porcentuales, con intervalos que incluyen cero (límite superior de 2.1 puntos en ambos).

En $k$ el resultado cambia. Por residencia, la desviación estándar es de 5.7 puntos (IC 95%: 2.0 a 10.7), mientras que por ocurrencia es cero (IC: 0 a 5.3). Las víctimas que residen en alcaldías centrales tienen mayor probabilidad de que su denuncia llegue a carpeta (Venustiano Carranza 83%, Benito Juárez 79%, Iztacalco 78%) que las que residen en la periferia (Cuajimalpa 50%, Tláhuac 60%). Esto sugiere que la barrera institucional varía con las características de la víctima más que con el lugar del hecho. 

## 10. Ruta administrativa

Si R fuera igual en todas las alcaldías, la tasa de carpetas debería ser aproximadamente proporcional a la victimización. Aquí comparo directamente las dos fuentes con R_adm = C / V: primero cómo varía entre alcaldías y después por qué su nivel es tan distinto del de R de encuesta.

### 10.1 Variación entre alcaldías

In [ ]:
# Ruta administrativa agrupada: R_adm = C / V, 2020–2024

# R_adm = C / V, ambas por 100 mil hab. 18+ y sumando los cinco años.
#   C:     carpetas núcleo con completitud, por lugar del hecho
#   V_ocu: incidentes ENVIPE ocurridos en la alcaldía (cualquier residencia)
#   V_res: incidentes ENVIPE de residentes de la alcaldía (cualquier lugar)
# C y V_ocu se cuentan por lugar del hecho y se dividen entre la misma
# población residente, así que la población flotante se cancela en C/V_ocu.
# Con V_res no se cancela.
#
# EMPAREJAMIENTO DE CATEGORÍAS: C núcleo solo incluye los seis tipos cuya
# definición en la Fiscalía tiene el mismo alcance que el reactivo de la
# ENVIPE. Comparar esas carpetas contra TODOS los delitos de la encuesta mezcla
# categorías. Por eso calculo también V_ocu_nuc y R de encuesta restringidas a
# esos mismos seis tipos: es la comparación limpia entre las dos fuentes.

corr = pd.read_csv(RAIZ / "datos" / "procesados" / "correspondencia_bpcod_fgj_v3.csv",
                   encoding="utf-8-sig", dtype={"decision_final": str, "bpcod_propuesto": str})
NUCLEO_BPCOD = sorted(corr.loc[corr["v_nucleo"], "decision_final"].unique())
print("Tipos ENVIPE del núcleo:", [f"{b} {ETIQUETAS[b]}" for b in NUCLEO_BPCOD], "\n")

# --- 1. Población no residente ---------------------------------------------------
ocu = UNIVERSO & inc["alc_ocu"].isin(ALC16)
g = inc[ocu].assign(no_res=lambda x: x["alc_res"] != x["alc_ocu"])
pct_nr = g.groupby("alc_ocu").apply(
    lambda x: (x["FAC_DEL"] * x["no_res"]).sum() / x["FAC_DEL"].sum(), include_groups=False)
geo["pct_no_residente"] = 100 * geo["alc"].map(pct_nr)

# --- 2. Victimización: todos los delitos y solo los del núcleo -----------------------
pob5 = datos["maestro"].groupby("cve_mun")["pob_18_conapo"].sum()        # 5 años
res = UNIV_RES & inc["alc_res"].isin(ALC16)
ocu_nuc = ocu & inc["BPCOD"].isin(NUCLEO_BPCOD)
tasa = lambda mascara, col: 1e5 * geo["alc"].map(inc[mascara].groupby(col)["FAC_DEL"].sum()) / geo["alc"].map(pob5)
geo["V_ocu"] = tasa(ocu, "alc_ocu")
geo["V_res"] = tasa(res, "alc_res")
geo["V_ocu_nuc"] = tasa(ocu_nuc, "alc_ocu")
geo["share_nucleo"] = geo["V_ocu_nuc"] / geo["V_ocu"]          # mezcla de delitos de la alcaldía

# --- 3. R administrativa y R de encuesta con categorías emparejadas -------------------
geo["tasa_C"] = geo["alc"].map(C5["tasa_C"])
geo["R_adm_ocu"] = geo["tasa_C"] / geo["V_ocu"]
geo["R_adm_res"] = geo["tasa_C"] / geo["V_res"]
geo["R_adm_nuc"] = geo["tasa_C"] / geo["V_ocu_nuc"]
geo["R_enc_nuc"] = [estimar(inc, UNIVERSO & (inc["alc_ocu"] == a) & inc["BPCOD"].isin(NUCLEO_BPCOD))["R"]
                    for a in geo["alc"]]
R_enc_nuc_ciudad = estimar(inc, UNIVERSO & inc["BPCOD"].isin(NUCLEO_BPCOD))["R"]

print("Razón máx/mín:", geo[["V_ocu", "V_ocu_nuc", "V_res", "tasa_C", "R_adm_nuc"]]
      .agg(lambda s: s.max() / s.min()).round(2).to_dict())
print(f"Mediana R_adm (todos los delitos): {geo['R_adm_ocu'].median():.4f}   "
      f"R encuesta, todos: {ciudad5['R']:.4f}")
print(f"Mediana R_adm (núcleo emparejado): {geo['R_adm_nuc'].median():.4f}   "
      f"R encuesta, núcleo: {R_enc_nuc_ciudad:.4f}")

print("\nSpearman (rho, p):")
for x, y in [("V_ocu", "tasa_C"), ("V_ocu_nuc", "tasa_C"), ("V_res", "tasa_C"),
             ("R_adm_nuc", "R_enc_nuc"), ("R_adm_nuc", "pct_no_residente"),
             ("share_nucleo", "pct_no_residente"), ("R_adm_res", "pct_no_residente")]:
    r_, p_ = spearmanr(geo[x], geo[y])
    print(f"  {x:12s} vs {y:16s}: {r_:5.2f}  (p = {p_:.3f})")

print("\nPor alcaldía, ordenado por % de víctimas no residentes:\n")
print(geo[["nombre", "pct_no_residente", "V_ocu_nuc", "tasa_C", "R_adm_nuc", "R_enc_nuc"]]
      .sort_values("pct_no_residente").round(3).to_string(index=False))

geo.to_csv(RUTA_SALIDA / "robustez_residencia_ruta_adm.csv", index=False, encoding="utf-8-sig")

### 10.2 Descomposición de la diferencia de nivel

In [ ]:
# ¿Por qué R_adm es ~5 veces menor que R de encuesta? Descomposición del nivel

# A nivel ciudad, con el mismo denominador V:
#   R_enc / R_adm = (C_enc / V) / (C_núcleo / V) = C_enc / C_núcleo
#   C_enc    = carpetas que IMPLICA la encuesta = Σ FAC_DEL × carpeta (5 años)
#   C_núcleo = carpetas de la Fiscalía en la variante núcleo (5 años)
#
# El factor se descompone EXACTAMENTE en dos partes que se multiplican:
#   (a) C_enc / C_enc_N      categorías: cuántas carpetas de encuesta caen en
#                            tipos FUERA del núcleo (N = los seis tipos del
#                            núcleo). Es un efecto de definición, no de medición.
#   (b) C_enc_N / C_núcleo   en los seis tipos con alcance equivalente: cuántas
#                            carpetas reporta la encuesta por cada carpeta que
#                            registra la Fiscalía. Aquí se mezclan el autorreporte
#                            de la víctima, la unidad de conteo (incidentes contra
#                            carpetas), el universo de cada fuente y el periodo.
# El término (b) se reporta también tipo por tipo.

enc = inc.loc[UNIVERSO, ["BPCOD", "FAC_DEL", "carpeta"]]
por_b = (enc.assign(wc=enc["FAC_DEL"] * enc["carpeta"])
         .groupby("BPCOD").agg(V=("FAC_DEL", "sum"), C_enc=("wc", "sum")))
por_b["C_fgj_nucleo"] = (corr[corr["v_nucleo"]].groupby("decision_final")["carpetas_2020_2024"].sum()
                         .reindex(por_b.index).fillna(0))
por_b["en_nucleo"] = por_b.index.isin(NUCLEO_BPCOD)
por_b["tipo"] = por_b.index.map(ETIQUETAS)
por_b["R_enc_pct"] = 100 * por_b["C_enc"] / por_b["V"]
por_b["R_adm_pct"] = 100 * por_b["C_fgj_nucleo"] / por_b["V"]
por_b["encuesta_por_carpeta_fgj"] = por_b["C_enc"] / por_b["C_fgj_nucleo"].replace(0, np.nan)

print("Tipos del núcleo, CDMX 2020–2024 (totales de cinco años):\n")
print(por_b[por_b["en_nucleo"]][["tipo", "V", "C_enc", "C_fgj_nucleo", "R_enc_pct", "R_adm_pct",
                                 "encuesta_por_carpeta_fgj"]].round(2).to_string())

# --- Descomposición exacta -------------------------------------------------------------
C_enc_tot = por_b["C_enc"].sum()
C_enc_N = por_b.loc[por_b["en_nucleo"], "C_enc"].sum()
C_nucleo = corr.loc[corr["v_nucleo"], "carpetas_2020_2024"].sum()
a, b = C_enc_tot / C_enc_N, C_enc_N / C_nucleo
print(f"\nFactor total  C_enc / C_núcleo = {C_enc_tot / C_nucleo:.2f}")
print(f"  (a) categorías fuera del núcleo                 : × {a:.2f}")
print(f"  (b) carpetas reportadas por carpeta registrada  : × {b:.2f}")
print(f"  Verificación a·b = {a * b:.2f}")

# --- Contexto del registro ----------------------------------------------------------------
total_fgj = corr["carpetas_2020_2024"].sum()
sin_equiv = corr.loc[corr["decision_final"].str.strip() == "—", "carpetas_2020_2024"].sum()
print(f"\nCarpetas de la Fiscalía 2020–2024: {total_fgj:,.0f}")
print(f"  en el núcleo:                      {100 * C_nucleo / total_fgj:5.1f}%")
print(f"  sin equivalente en la ENVIPE:      {100 * sin_equiv / total_fgj:5.1f}%")
print(f"\nC_núcleo según correspondencia: {C_nucleo:,.0f}   según maestro: "
      f"{datos['maestro']['C_nucleo'].sum():,.0f}  (difieren por carpetas sin alcaldía)")

por_b.to_csv(RUTA_SALIDA / "descomposicion_nivel_R_adm.csv", encoding="utf-8-sig")

*Pendiente:* interpretación de la variación de R_adm entre alcaldías y de la descomposición del nivel.

## 11. Contraste externo con la ENSU

Comparo mi victimización con la ENSU, una encuesta independiente de la ENVIPE con muestra propia por alcaldía. Solo sirven las ediciones de junio y diciembre, que preguntan por la victimización del hogar en los últimos seis meses. En las de marzo y septiembre, las variables `BP1_6_*` existen pero preguntan otra cosa.

In [ ]:
# Inventario de la ENSU: qué ediciones sirven y qué versión del cuestionario usan

# Cuento cuántos reactivos de victimización del hogar trae cada edición, en
# lugar de exigir ocho: el cuestionario cambió y las ediciones anteriores
# traen menos delitos. Uso el nombre exacto BP1_6_n con UN dígito, porque en
# los trimestres I y III existen BP1_6_01 … BP1_6_13 con otro significado.
# También listo las variables de diseño que encuentro, porque en algunas
# ediciones pueden llamarse distinto.

import zipfile, re, tempfile
from pathlib import PurePath

try:
    from dbfread import DBF                     # para la edición 2020 en .dbf
except ImportError:
    DBF = None
    print("Aviso: falta dbfread (pip install dbfread); las ediciones .dbf no se leerán.\n")

RAIZ_ENSU = RAIZ / "datos" / "crudos" / "ENSU"
PATRON_CB = re.compile(r"cb[_\-]?(\d{2})(\d{2})", re.I)       # ENSU_CB_mmaa
PATRON_VICT = re.compile(r"^BP1_6_(\d)$")                     # victimización: un dígito
PREFIJOS_DISENO = ("FAC", "EST", "UPM", "CVE", "ENT", "MUN", "CD")

def leer_csv(abrir, **kw):
    """Lee un CSV probando UTF-8 y luego Latin-1 (el INEGI usa ambos)."""
    for enc in ("utf-8", "latin-1"):
        try:
            with abrir() as f:
                return pd.read_csv(f, encoding=enc, dtype=str, **kw)
        except UnicodeDecodeError:
            continue
    raise ValueError("No se pudo leer con UTF-8 ni Latin-1")

def leer_dbf(arch, miembro, columnas=None):
    """Lee un .dbf (suelto o dentro de un .zip). Sin columnas, solo devuelve los nombres."""
    with tempfile.TemporaryDirectory() as tmp:
        ruta = zipfile.ZipFile(arch).extract(miembro, tmp) if miembro else arch
        tabla = DBF(ruta, encoding="latin-1", load=False)
        if columnas is None:
            return pd.DataFrame(columns=tabla.field_names)
        return pd.DataFrame([{c: str(r[c]) for c in columnas} for r in tabla])

# --- 1. Localizar las tablas CB, sueltas o dentro de .zip ------------------------
fuentes = []
for arch in sorted(RAIZ_ENSU.rglob("*")):
    if arch.suffix.lower() == ".zip":
        with zipfile.ZipFile(arch) as z:
            fuentes += [(arch, m) for m in z.namelist()
                        if PATRON_CB.search(PurePath(m).name) and m.lower().endswith((".csv", ".dbf"))]
    elif arch.suffix.lower() in (".csv", ".dbf") and PATRON_CB.search(arch.name):
        fuentes.append((arch, None))

# --- 2. Revisar cada edición -----------------------------------------------------
filas = []
for arch, miembro in fuentes:
    nombre = PurePath(miembro).name if miembro else arch.name
    mes, aa = PATRON_CB.search(nombre).groups()
    formato = nombre.rsplit(".", 1)[-1].lower()
    fila = {"anio": 2000 + int(aa), "mes": mes, "formato": formato,
            "fuente": str(arch.relative_to(RAIZ_ENSU)) + (f" :: {miembro}" if miembro else "")}

    if formato == "dbf" and DBF is None:
        filas.append(fila)
        continue
    if formato == "csv":
        abrir = ((lambda a=arch, m=miembro: zipfile.ZipFile(a).open(m)) if miembro
                 else (lambda a=arch: open(a, "rb")))
        cab = leer_csv(abrir, nrows=0).columns
    else:
        cab = leer_dbf(arch, miembro).columns

    mayus = {c.upper(): c for c in cab}
    vict = sorted(int(m.group(1)) for c in mayus if (m := PATRON_VICT.match(c)))
    fila["reactivos_vict"] = len(vict)
    fila["variables_diseno"] = sorted(c for c in mayus if c.startswith(PREFIJOS_DISENO))

    # Cuestionarios de la CDMX y valores del primer reactivo, si la edición sirve
    if vict and "CVE_ENT" in mayus:
        cols = [mayus["CVE_ENT"], mayus["BP1_6_1"]]
        df = leer_csv(abrir, usecols=cols) if formato == "csv" else leer_dbf(arch, miembro, cols)
        cdmx = df[mayus["CVE_ENT"]].str.strip().str.zfill(2) == "09"
        fila["cuestionarios_cdmx"] = int(cdmx.sum())
        fila["valores_BP1_6_1"] = df.loc[cdmx, mayus["BP1_6_1"]].str.strip().value_counts().to_dict()
    filas.append(fila)

inventario_ensu = pd.DataFrame(filas).sort_values(["anio", "mes"]).reset_index(drop=True)
print(inventario_ensu.drop(columns="variables_diseno").to_string())

print("\nVariables de diseño por edición:")
for _, r in inventario_ensu.iterrows():
    print(f"  {r['anio']}-{r['mes']}: {r.get('variables_diseno')}")

print("\nEdiciones con victimización del hogar (junio o diciembre):")
print(inventario_ensu.query("mes in ['06', '12'] and reactivos_vict > 0")
      [["anio", "mes", "reactivos_vict", "cuestionarios_cdmx"]].to_string(index=False))

In [ ]:
# Revisión de una edición anterior: variables que trae la tabla CB de junio de 2024
z = zipfile.ZipFile(RAIZ_ENSU / "ensu_bd_2024_csv" / "ensu_bd_junio_2024_csv.zip")
m = [n for n in z.namelist() if "CB" in n.upper() and n.lower().endswith(".csv")][0]
cols = leer_csv(lambda: z.open(m), nrows=0).columns
print(m, len(cols)); print([c for c in cols if c.upper().startswith(("BP1", "FAC", "EST", "UPM", "CVE"))])

*Pendiente:* la edición de junio de 2024 trae `BP1_6_1` a `BP1_6_6`, seis delitos y no ocho, y el inventario la descartó porque buscaba ocho. Antes de estimar reviso qué delitos cubre cada versión del cuestionario y ajusto el inventario.

In [ ]:
# Prevalencia de hogares victimizados por alcaldía en la ENSU, 2020–2024

# EDICIONES: cada año de referencia t lo cubren junio t (dic t−1 a may t) y
# diciembre t (jun a nov t). Para 2020 solo existe diciembre (cubre medio año).
# Uso únicamente junio y diciembre: en marzo y septiembre los nombres BP1_6_n
# preguntan otra cosa (en marzo de 2020 el inventario encontró nueve).
#
# MEDIDA: el hogar es víctima si contestó "sí" (1) a alguno de los seis
# reactivos comunes a todas las ediciones. En las ediciones con ocho ignoro
# los dos fraudes, que las anteriores no preguntan. Si un hogar no tiene
# ningún "sí" pero sí algún "no sabe" (9), su condición es desconocida y lo
# excluyo.
#
# FACTOR: FAC_VIV, porque la pregunta es sobre el hogar. Donde la tabla CB no
# lo trae (diciembre de 2020 y 2021), lo tomo de la tabla de la vivienda de la
# misma edición, uniendo por la llave de la vivienda.
#
# VARIANZA: la misma linealización de Taylor que en la ENVIPE, con el estrato
# definido como edición + EST_DIS: cada edición es una muestra independiente.
# Agrupar las nueve ediciones da la prevalencia semestral promedio 2020–2024.

EDICIONES = [(2020, "12")] + [(a, m) for a in range(2021, 2025) for m in ("06", "12")]
VICT6 = [f"BP1_6_{i}" for i in range(1, 7)]
LLAVE_VIV = ["UPM", "PRO_VIV", "VIV_SEL"]

def buscar_tabla(anio, mes, tabla):
    """(archivo .zip, miembro) de la tabla ENSU_<tabla>_mmaa."""
    patron = re.compile(rf"{tabla}[_\-]?{mes}{str(anio)[2:]}\.(csv|dbf)$", re.I)
    for arch in sorted(RAIZ_ENSU.rglob("*.zip")):
        with zipfile.ZipFile(arch) as z:
            for m in z.namelist():
                if patron.search(PurePath(m).name):
                    return arch, m
    raise FileNotFoundError(f"No encontré ENSU_{tabla}_{mes}{str(anio)[2:]}")

def leer_tabla(arch, miembro, columnas):
    """Lee solo las columnas pedidas que existan (sin distinguir mayúsculas).
    Devuelve los nombres en mayúsculas y los valores como texto limpio."""
    if miembro.lower().endswith(".csv"):
        abrir = lambda: zipfile.ZipFile(arch).open(miembro)
        cab = leer_csv(abrir, nrows=0).columns
        mayus = {c.upper(): c for c in cab}
        presentes = [mayus[c] for c in columnas if c in mayus]
        df = leer_csv(abrir, usecols=presentes)
    else:
        cab = leer_dbf(arch, miembro).columns
        mayus = {c.upper(): c for c in cab}
        presentes = [mayus[c] for c in columnas if c in mayus]
        df = leer_dbf(arch, miembro, presentes)
    df.columns = [c.upper() for c in df.columns]
    return df.apply(lambda s: s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True))

partes, control = [], []
for anio, mes in EDICIONES:
    arch, miembro = buscar_tabla(anio, mes, "CB")
    cb = leer_tabla(arch, miembro, ["ID_VIV", "CVE_ENT", "CVE_MUN", *LLAVE_VIV, "EST_DIS", "UPM_DIS", "FAC_VIV", *VICT6])
    cb = cb[cb["CVE_ENT"].str.zfill(2) == "09"].copy()
    assert all(v in cb for v in VICT6), f"{anio}-{mes}: faltan reactivos de victimización"

    # Factor de vivienda desde la tabla VIV si la CB no lo trae
    origen_fac = "CB"
    if "FAC_VIV" not in cb:
        arch_v, miembro_v = buscar_tabla(anio, mes, "VIV")
        viv = leer_tabla(arch_v, miembro_v, ["ID_VIV", *LLAVE_VIV, "FAC_VIV"])
        # ID_VIV identifica la vivienda en todas las ediciones; UPM + VIV_SEL no basta
        # cuando falta PRO_VIV (en diciembre de 2020 repite 107 viviendas)
        llave = (["ID_VIV"] if "ID_VIV" in cb and "ID_VIV" in viv
                 else [k for k in LLAVE_VIV if k in cb and k in viv])
        assert not viv.duplicated(llave).any(), f"{anio}-{mes}: llave de vivienda repetida en VIV"
        cb = cb.merge(viv[llave + ["FAC_VIV"]], on=llave, how="left", validate="many_to_one")
        assert cb["FAC_VIV"].notna().all(), f"{anio}-{mes}: viviendas sin FAC_VIV tras la unión"
        origen_fac = "VIV (" + "+".join(llave) + ")"

    # Condición de victimización del hogar
    v = cb[VICT6]
    si = (v == "1").any(axis=1)
    desconocido = ~si & (v == "9").any(axis=1)
    llave_cb = ["ID_VIV"] if "ID_VIV" in cb else [k for k in LLAVE_VIV if k in cb]
    control.append({"edicion": f"{anio}-{mes}", "hogares_cdmx": len(cb),
                    "llave_repetida": int(cb.duplicated(llave_cb).sum()),
                    "desconocidos": int(desconocido.sum()), "factor_desde": origen_fac})

    cb = cb[~desconocido]
    estrato = f"{anio}{mes}_" + cb["EST_DIS"]
    partes.append(pd.DataFrame({
        "edicion": f"{anio}-{mes}",
        "alc": cb["CVE_MUN"].str.zfill(3).to_numpy(),
        "estrato": estrato.to_numpy(),
        "upm_diseno": (estrato + "_" + cb["UPM_DIS"]).to_numpy(),
        "FAC_VIV": cb["FAC_VIV"].astype(float).to_numpy(),
        "victima": si[~desconocido].to_numpy(),
        "todos": True}))

ensu = pd.concat(partes, ignore_index=True)
print("Control por edición:\n")
print(pd.DataFrame(control).to_string(index=False))

# --- Prevalencia por edición (ciudad) y por alcaldía (nueve ediciones agrupadas) ----
por_edicion = pd.DataFrame([{"edicion": e, **razon_taylor(g, pd.Series(True, index=g.index),
                                                          "victima", "todos", peso="FAC_VIV")}
                            for e, g in ensu.groupby("edicion")])
print("\nPrevalencia semestral de hogares victimizados, CDMX (%):\n")
print(por_edicion.assign(est=100 * por_edicion["est"], ee=100 * por_edicion["ee"])
      .round(2).to_string(index=False))

ensu_alc = pd.DataFrame([{"alc": a, "nombre": NOMBRES[a],
                          **razon_taylor(ensu, ensu["alc"] == a, "victima", "todos", peso="FAC_VIV")}
                         for a in ALC16]).rename(columns={"est": "prev_ensu", "ee": "ee_prev_ensu", "n": "n_hogares"})
print("\nPrevalencia semestral promedio 2020–2024 por alcaldía de residencia (%):\n")
print(ensu_alc.assign(prev_ensu=100 * ensu_alc["prev_ensu"], ee_prev_ensu=100 * ensu_alc["ee_prev_ensu"])
      .sort_values("prev_ensu", ascending=False).round(2).to_string(index=False))

ensu_alc.to_csv(RUTA_SALIDA / "ensu_prevalencia_alcaldia.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Contraste externo: ENSU contra la ENVIPE construida igual

# LA MISMA MEDIDA EN LAS DOS ENCUESTAS
#   Geografía: alcaldía de RESIDENCIA (la ENSU solo pregunta a residentes).
#   Delitos:   los seis comunes (BPCOD 01, 02, 04, 05, 06 y 09).
#   Periodo:   2020–2024.
# Diferencia que no puedo eliminar: la ENSU mide PREVALENCIA (% de hogares
# con al menos un delito en el semestre) y mi archivo de la ENVIPE solo trae
# incidentes, así que del lado ENVIPE uso la TASA de incidentes por 100 mil
# hab. 18+. Por eso comparo ÓRDENES (Spearman), no niveles.
#
# INCERTIDUMBRE: las dos medidas tienen error muestral. Simulo cada una con
# su EE de diseño y recalculo el Spearman 2,000 veces para obtener un
# intervalo. Además calculo la FIABILIDAD de cada medida entre alcaldías
# (fracción de su varianza que no es ruido muestral): la correlación máxima
# que dos medidas ruidosas pueden alcanzar es √(fiab_1 · fiab_2).
#
# Como referencia secundaria repito el contraste con la ENVIPE por lugar de
# OCURRENCIA, que es la geografía principal de mi trabajo.

BPCOD_ENSU = ["01", "02", "04", "05", "06", "09"]
pob5 = datos["maestro"].groupby("cve_mun")["pob_18_conapo"].sum()

def tasa_con_ee(mascara, col_alc):
    """Tasa de incidentes por 100 mil hab. 18+ (5 años) con EE de diseño del total."""
    filas = []
    for a in ALC16:
        z = (inc["FAC_DEL"] * (mascara & (inc[col_alc] == a))).to_numpy()
        filas.append({"alc": a, "tasa": 1e5 * z.sum() / pob5[a],
                      "ee_tasa": 1e5 * np.sqrt(var_diseno(inc, z)) / pob5[a]})
    return pd.DataFrame(filas).set_index("alc")

env_res = tasa_con_ee(UNIV_RES & inc["BPCOD"].isin(BPCOD_ENSU), "alc_res")
env_ocu = tasa_con_ee(UNIVERSO & inc["BPCOD"].isin(BPCOD_ENSU), "alc_ocu")

contraste_ensu = (ensu_alc.set_index("alc")[["nombre", "prev_ensu", "ee_prev_ensu"]]
                  .join(env_res.add_suffix("_res")).join(env_ocu.add_suffix("_ocu")))

def fiabilidad(est, ee):
    """1 − (ruido muestral promedio) / (varianza total), en escala log."""
    x = np.log(est); ruido = np.mean((ee / est) ** 2)
    return max(0.0, 1 - ruido / np.var(x, ddof=1))

rng_e = np.random.default_rng(20261014)
B = 2000
print("Spearman ENSU (prevalencia) vs ENVIPE (tasa), seis delitos comunes, 2020–2024:\n")
filas_c = []
for geo_env in ["res", "ocu"]:
    e, se = contraste_ensu["prev_ensu"].to_numpy(), contraste_ensu["ee_prev_ensu"].to_numpy()
    t, st = contraste_ensu[f"tasa_{geo_env}"].to_numpy(), contraste_ensu[f"ee_tasa_{geo_env}"].to_numpy()
    rho = spearmanr(e, t)[0]
    sims = [spearmanr(e + rng_e.normal(0, se), t + rng_e.normal(0, st))[0] for _ in range(B)]
    f_e, f_t = fiabilidad(e, se), fiabilidad(t, st)
    filas_c.append({"envipe_por": "residencia" if geo_env == "res" else "ocurrencia",
                    "spearman": rho, "ic95_inf": np.quantile(sims, 0.025), "ic95_sup": np.quantile(sims, 0.975),
                    "fiab_ensu": f_e, "fiab_envipe": f_t, "corr_maxima_posible": np.sqrt(f_e * f_t)})
resumen_ensu = pd.DataFrame(filas_c)
print(resumen_ensu.round(3).to_string(index=False))

# Señal territorial de la ENSU con el mismo estimador REML que uso para tipos y alcaldías
disp_ensu = resumen_dispersion(contraste_ensu["prev_ensu"], contraste_ensu["ee_prev_ensu"], "ensu_alcaldias", "prev")
print(f"\nENSU: DE de la señal entre alcaldías = {disp_ensu['de_pp']:.1f} pp "
      f"(IC 95%: {disp_ensu['de_pp_ic_inf']:.1f} a {disp_ensu['de_pp_ic_sup']:.1f}); "
      f"razón máx/mín = {contraste_ensu['prev_ensu'].max() / contraste_ensu['prev_ensu'].min():.2f}")
print(f"ENVIPE residencia: razón máx/mín de la tasa = "
      f"{contraste_ensu['tasa_res'].max() / contraste_ensu['tasa_res'].min():.2f}")

vista = contraste_ensu.assign(
    rango_ensu=contraste_ensu["prev_ensu"].rank(ascending=False).astype(int),
    rango_envipe_res=contraste_ensu["tasa_res"].rank(ascending=False).astype(int),
    rango_envipe_ocu=contraste_ensu["tasa_ocu"].rank(ascending=False).astype(int),
    prev_ensu=100 * contraste_ensu["prev_ensu"]).sort_values("rango_ensu")
print("\nOrden de las alcaldías en cada fuente:\n")
print(vista[["nombre", "prev_ensu", "rango_ensu", "tasa_res", "rango_envipe_res", "tasa_ocu", "rango_envipe_ocu"]]
      .round(1).to_string())

contraste_ensu.to_csv(RUTA_SALIDA / "contraste_ensu_envipe.csv", encoding="utf-8-sig")

**Lo que leo en el contraste con la ENSU.** Construí la misma medida en las dos encuestas: residencia, los seis delitos comunes y el periodo 2020–2024. Las dos coinciden en que la victimización varía entre alcaldías y en la magnitud de esa variación: en la ENSU la señal entre alcaldías tiene una desviación estándar de 4.7 puntos (IC 95%: 3.3 a 7.1) y una razón máx/mín de 1.9, y en la ENVIPE la razón es de 2.3. No coinciden, en cambio, en el orden de las alcaldías: la correlación de rangos es 0.29 (IC 95%: 0.04 a 0.51) con la ENVIPE por residencia y 0.33 con la ENVIPE por ocurrencia, muy por debajo del 0.90 que permitiría la fiabilidad de las dos medidas. El ruido muestral no explica la discrepancia.

Atribuyo la diferencia a tres rasgos de construcción que no puedo igualar con los archivos disponibles: la ENSU mide prevalencia de hogares victimizados y la ENVIPE cuenta incidentes, incluidas las repeticiones; en robo en calle, otro tipo de robo y extorsión, la ENVIPE pregunta solo por la persona seleccionada, mientras que la ENSU pregunta por cualquier integrante del hogar; y el periodo de recuerdo es de seis meses en la ENSU y de un año en la ENVIPE.

Este contraste no afecta el resultado sobre R, porque la ENSU no pregunta por denuncia. Sí acota la interpretación de la ruta administrativa: la victimización varía entre alcaldías en una magnitud comparable según las dos fuentes, pero su orden territorial depende de la fuente y de la forma de medirla, de modo que no atribuyo la desigualdad de las carpetas a un orden específico de incidencia.

### 9.2 ¿La señal en k por residencia se repite año con año?

In [ ]:
# ¿El k por residencia se repite año con año?

# La señal territorial en k por residencia salió de UNA prueba sobre datos
# agrupados 2020–2024. Una diferencia territorial real debería repetirse en
# los distintos años; una producida por el azar de la muestra, no. La pongo a
# prueba de dos formas, y en las dos aplico exactamente lo mismo a la
# geografía de ocurrencia como referencia:
#
# 1. MODELO DE DOS COMPONENTES (el mismo motor de la sección de potencia):
#      θ_at = μ_t + u_a + v_at + e_at,   θ = arcsen√k
#    con k directa por alcaldía y año, ψ_at = EE² / (4k(1−k)) (método delta).
#    Si la diferencia entre alcaldías es persistente, σ²_alc > 0 y la prueba
#    de razón de verosimilitudes lo detecta. Excluyo celdas con menos de 10
#    denuncias o con k en 0 o en 1 (su ψ no está definida).
#
# 2. MITADES INDEPENDIENTES: k agrupada en 2020, 2022 y 2024 contra k agrupada
#    en 2021 y 2023. Son muestras distintas: si el orden de las alcaldías se
#    repite entre mitades, la diferencia no es azar.

GEOGRAFIAS = {"residencia": (UNIV_RES, "alc_res"), "ocurrencia": (UNIVERSO, "alc_ocu")}

# --- k directa por alcaldía, año y geografía ----------------------------------------
filas = []
for geo_nombre, (univ, col) in GEOGRAFIAS.items():
    for anio, g in inc.groupby("anio_ref"):
        for a in ALC16:
            dom = (univ & (inc[col] == a)).loc[g.index]
            with np.errstate(invalid="ignore", divide="ignore"):
                r = razon_taylor(g, dom, "carpeta", "denuncio")
            filas.append({"geografia": geo_nombre, "anio_ref": anio, "alc": a,
                          "k": r["est"], "ee": r["ee"], "n": r["n"]})
k_anual = pd.DataFrame(filas)

print("k por residencia, por año (%), y denuncias en la muestra:\n")
tabla_k = (k_anual.query("geografia == 'residencia'")
           .assign(nombre=lambda x: x["alc"].map(NOMBRES), k=lambda x: (100 * x["k"]).round(0))
           .pivot(index="nombre", columns="anio_ref", values="k"))
tabla_k["n_total"] = (k_anual.query("geografia == 'residencia'")
                      .assign(nombre=lambda x: x["alc"].map(NOMBRES)).groupby("nombre")["n"].sum())
print(tabla_k.sort_values(2020).to_string())

# --- 1. Modelo de dos componentes ---------------------------------------------------
def prueba_persistencia(d):
    d = d[(d["k"] > 0) & (d["k"] < 1) & (d["ee"] > 0) & (d["n"] >= 10)].reset_index(drop=True)
    y = np.arcsin(np.sqrt(d["k"].to_numpy()))
    psi = d["ee"].to_numpy() ** 2 / (4 * d["k"].to_numpy() * (1 - d["k"].to_numpy()))
    anios = sorted(d["anio_ref"].unique())
    X = np.column_stack([np.ones(len(d))] + [(d["anio_ref"] == a).to_numpy(float) for a in anios[1:]])
    Z = pd.get_dummies(d["alc"], dtype=float).to_numpy()
    r = ajustar_dos_componentes(y, X, psi, Z @ Z.T)
    mu = np.arcsin(np.sqrt(np.average(d["k"], weights=1 / psi)))
    return {"celdas": len(d), "s2_alc": r["s2_alc"], "s2_cel": r["s2_cel"],
            "frac_persistente": r["s2_alc"] / (r["s2_alc"] + r["s2_cel"]) if r["s2_alc"] + r["s2_cel"] > 0 else 0.0,
            "de_persistente_pp": 100 * np.sin(2 * mu) * np.sqrt(r["s2_alc"]),
            "lrt": r["lrt"], "p_valor": r["p_valor"]}

persistencia_k = pd.DataFrame([{"geografia": geo_nombre, **prueba_persistencia(d)}
                               for geo_nombre, d in k_anual.groupby("geografia")])
print("\nModelo alcaldía + celda en k, con efectos de año:\n")
print(persistencia_k.round(5).to_string(index=False))

# --- 2. Mitades independientes ------------------------------------------------------
MITAD_A, MITAD_B = [2020, 2022, 2024], [2021, 2023]
filas_m = []
for geo_nombre, (univ, col) in GEOGRAFIAS.items():
    ka = [estimar(inc, univ & (inc[col] == a) & inc["anio_ref"].isin(MITAD_A))["k"] for a in ALC16]
    kb = [estimar(inc, univ & (inc[col] == a) & inc["anio_ref"].isin(MITAD_B))["k"] for a in ALC16]
    rho, p = spearmanr(ka, kb)
    filas_m.append({"geografia": geo_nombre, "spearman_mitades": rho, "p_valor": p})
mitades_k = pd.DataFrame(filas_m)
print("\nOrden de las alcaldías en k: 2020+2022+2024 contra 2021+2023:\n")
print(mitades_k.round(3).to_string(index=False))

k_anual.to_csv(RUTA_SALIDA / "k_por_alcaldia_anio_geografia.csv", index=False, encoding="utf-8-sig")

### IV. Resultados

### A. Cifras de ciudad

De cada 100 delitos ocurridos en la Ciudad de México entre 2020 y 2024, 11 se denunciaron y 7 quedaron registrados en una carpeta de investigación. La probabilidad de denuncia, $d$, osciló entre 10.0% y 11.8% según el año (error estándar de 0.8 a 0.9 puntos). La probabilidad de que una denuncia se convierta en carpeta, $k$, osciló entre 64.6% y 68.5% (error estándar de 3.1 a 4.4 puntos), y la probabilidad de registro, $R$, entre 6.8% y 7.8%. Ninguna diferencia entre años excede 1.5 errores estándar.

La pérdida se concentra en una etapa. De cada 100 delitos, 89 quedan fuera del registro en la barrera ciudadana y 4 más en la institucional. Aun así, una de cada tres denuncias no se convierte en carpeta, pese a que el Código Nacional de Procedimientos Penales obliga al ministerio público a iniciarla. Esta cifra proviene del autorreporte de la víctima; al compararla con el registro de la Fiscalía, la encuesta implica casi tres carpetas por cada carpeta registrada en los tipos de delito comparables (Sección F), de modo que la pérdida en la barrera institucional debe leerse como una cota inferior.

### B. Ausencia de variación territorial y su potencia

No se detectó variación persistente de la probabilidad de registro entre alcaldías. En el modelo con efecto de alcaldía y efecto de celda, sin covariables, la fracción persistente de la variación no muestral es 19% en $d$ ($p = 0.29$), 12% en $R$ ($p = 0.36$) y cero en $k$. Con covariables, es cero en los tres parámetros.

Para saber si el método podía detectar esa variación, se simularon mundos en los que el parámetro reproduce una fracción $\lambda$ de la desigualdad de la tasa de carpetas, con el ruido muestral y la variación interanual observados. La prueba de razón de verosimilitudes tiene un tamaño empírico de 0.03 a 0.06. Si $R$ variara 3.5 a 1 con el orden de las carpetas, la prueba lo habría detectado en el 100% de las réplicas (99% con orden aleatorio). Alcanza 80% de potencia con brechas de 2.1 a 2.4 a 1 en $R$ y de 1.8 a 2.0 a 1 en $d$.

Los datos son compatibles con diferencias moderadas en $R$, de hasta unas 2 veces, lo que coincide con la cota superior por verosimilitud perfil. No son compatibles con que $R$ explique la mayor parte de la desigualdad del registro. La Fig. 1 lo muestra: las regiones de confianza de las 16 alcaldías se enciman en el plano $d$–$k$, y el orden de $R$ no guarda relación con el de la tasa de carpetas (Spearman 0.06, $p = 0.82$), que varía 3.5 a 1.

Dos límites acotan este resultado. Las comparaciones de un solo año carecen de potencia: distinguir dos alcaldías exigiría una razón de 4.9 a 1 en $R$. Por eso los traslapes anuales no se usan como evidencia. En $k$, 80% de potencia requiere una brecha de 26 a 32 puntos porcentuales. La encuesta no permite evaluar diferencias territoriales en la barrera institucional; solo su nivel, que en ninguna alcaldía se acerca al 100%.

El resultado nulo en $d$ y $R$ se sostiene al asignar los incidentes a la alcaldía de residencia de la víctima: la desviación estándar de la señal entre alcaldías es de 0.9 y 1.1 puntos, con intervalos que incluyen cero. En $k$ por residencia aparece un indicio de variación persistente, equivalente a una desviación estándar de 6.7 puntos ($p = 0.036$), que no se repite de forma significativa entre mitades independientes del periodo (Spearman 0.38, $p = 0.15$) y no sobrevive a una corrección por comparaciones múltiples. Se plantea como hipótesis: la barrera institucional podría depender de dónde vive la víctima más que de dónde ocurrió el delito.

### C. Heterogeneidad por tipo de delito

La probabilidad de denuncia varía 23 veces entre tipos de delito: de 3.3% en hostigamiento o abuso sexual a 74.7% en robo total de vehículo (Tabla I). Aun sin el robo de vehículo, va de 3.3% a 25.7% en robo a casa habitación.

El robo de vehículo funciona como control positivo, porque denunciar es requisito para cobrar el seguro. Su $d$ es 7.4 veces la del resto de los delitos ($z = 27$), se mantiene entre 68% y 83% los cinco años y es pareja entre alcaldías: de 65% a 90%, sin heterogeneidad detectable (Q de Cochran, $p = 0.52$). La medición separa lo que debe separar.

Para comparar tipos y alcaldías con el mismo criterio se estimó, en cada partición, la varianza de la señal entre categorías por REML, descontando el error muestral. En $d$, la desviación estándar entre tipos es de 16.8 puntos (7.7 sin vehículo) y entre alcaldías de 0.6. Aun con el límite inferior de los tipos y el superior de las alcaldías, la variación entre tipos es al menos 5.5 veces la territorial en $d$ (2.75 sin vehículo) y 4.9 veces en $R$ (2.25). La visibilidad del delito depende de qué le pasó a la víctima mucho más que de dónde ocurrió.

En $k$ la comparación no es concluyente: el cociente conservador es 0.95. Con todo, el robo de vehículo tiene $k$ de 84.1%, 19.6 puntos más que el resto ($z = 6.7$). Cuando la víctima necesita la carpeta, la barrera institucional se reduce. Violación sexual (38 casos) y secuestro (17) no tienen muestra suficiente para estimarse.

**Tabla I.** Parámetros por tipo de delito, CDMX, 2020–2024 (%). IC 95% en escala logit. \* Con reservas (CV > 30%, criterio INEGI). El valor de $k$ en Otros delitos se basa en 18 denuncias.

| Tipo de delito | $d$ [IC 95%] | $k$ | $R$ |
| --- | --- | --- | --- |
| Robo total de vehículo | 74.7 [69.7, 79.0] | 84.1 | 62.8 |
| Robo a casa habitación | 25.7 [22.6, 29.1] | 66.6 | 17.1 |
| Lesiones | 21.7 [17.0, 27.2] | 72.3 | 15.7 |
| Otros delitos | 20.1 [12.0, 31.8] | 79.8 | 16.1 |
| Amenazas verbales | 19.9 [16.9, 23.3] | 68.3 | 13.6 |
| Otro tipo de robo | 16.4 [12.0, 22.1] | 78.3 | 12.9 |
| Robo de accesorios o autopartes | 10.3 [9.2, 11.6] | 59.8 | 6.2 |
| Robo en calle o transporte | 9.2 [8.0, 10.5] | 59.9 | 5.5 |
| Extorsión | 6.3 [5.1, 7.8] | 58.1 | 3.7 |
| Fraude bancario | 5.9 [4.6, 7.4] | 68.2 | 4.0 |
| Fraude al consumidor | 5.1 [3.8, 6.8] | 56.8 | 2.9 |
| Hostigamiento o abuso sexual | 3.3 [1.9, 5.5] | 83.7 | 2.7\* |

### D. Mapa corregido

Corregir la tasa de carpetas por la probabilidad de registro conserva la magnitud de la desigualdad territorial y su extremo superior. Con las carpetas del núcleo, el periodo 2020–2024 agrupado y el factor de completitud de 2024, la razón entre la alcaldía más alta y la más baja pasa de 3.52 a 3.41. La correlación de rangos entre el orden observado y el corregido es 0.87 (límite inferior 0.78), con 5,000 simulaciones Monte Carlo de $R$ a partir de su error cuadrático medio. Cuauhtémoc queda primera y Cuajimalpa última en el 100% de las simulaciones (Fig. 2).

La corrección reacomoda la zona media, donde las alcaldías de los lugares 7 a 14 tienen tasas dentro de un rango de 20%. Coyoacán baja del lugar 8 al 13 (probabilidad de moverse más de dos lugares: 0.89) y Tlalpan del 12 al 15 (0.55), ambas con $R$ alta. Milpa Alta sube del 15 al 11 (0.60). Estos movimientos descansan en diferencias de $R$ dentro del rango que la Sección B mostró indetectable, por lo que no deben leerse como hallazgos sobre esas alcaldías.

Los mapas anuales se mueven más: la correlación de rangos baja a 0.61 en 2024 (límite inferior 0.32). La variación de $R$ de un año a otro no es persistente, y corregir cada año con su propia $R$ la traslada al mapa. El mapa corregido de referencia es el agrupado.

### E. Sensibilidad

Ninguna de las 12 combinaciones de cuatro definiciones de carpetas y tres fuentes de $R$ produce un mapa cualitativamente distinto del observado (Tabla II). La razón corregida entre extremos va de 2.4 a 4.9, contra 2.5 a 5.0 observada, y Cuauhtémoc conserva el primer lugar (en rango mediano) en las 12.

El modelo de $R$ incluye covariables derivadas de registros administrativos, lo que abre la posibilidad de circularidad. Por eso la corrección se repitió con la $R$ directa agrupada, sin modelo ni covariables. Con ella el orden se mueve más, como se espera de un estimador con más ruido, pero en la misma dirección: Coyoacán y Tlalpan vuelven a bajar (lugares 14 y 16). Los movimientos de la Sección D no son un artefacto de las covariables. El último lugar de Cuajimalpa, en cambio, depende de la fuente: con la $R$ directa sube al lugar 13.

Con $R$ fija en el promedio de la ciudad, el orden corregido es idéntico al observado por construcción. Es la referencia contra la que se miden las demás.

**Tabla II.** Correlación de rangos entre el orden observado y el corregido, 2020–2024 (mediana de 5,000 simulaciones).

| Definición de carpetas | Razón observada | $R$ modelada | $R$ directa | $R$ fija |
| --- | --- | --- | --- | --- |
| Núcleo | 3.52 | 0.87 | 0.70 | 1 |
| Núcleo, alta confianza | 3.49 | 0.84 | 0.67 | 1 |
| Núcleo y violencia familiar | 2.46 | 0.79 | 0.56 | 1 |
| Amplio | 5.00 | 0.89 | 0.73 | 1 |

### F. Contraste con el registro administrativo y con la ENSU

Con categorías emparejadas, es decir, los seis tipos de delito cuya definición en la Fiscalía tiene el mismo alcance que el reactivo de la encuesta, la probabilidad de registro administrativa, $R_{adm} = C/V$, es de 2.8% en la mediana de las alcaldías, contra 8.0% según la encuesta. El factor total de 5.2 entre ambas fuentes se descompone exactamente en dos partes: 1.77 de definición, por las carpetas que reporta la encuesta en tipos fuera del núcleo, y 2.94 de medición: en los tipos emparejados, la encuesta implica casi tres carpetas por cada carpeta registrada. Este segundo término combina autorreporte, unidad de conteo, diferencias de universo y periodo; por tipo va de 0.9 en violación sexual, donde la Fiscalía incluye víctimas menores de edad, a 5.3 en lesiones.

Entre alcaldías, $R_{adm}$ varía 2.3 a 1 y aumenta con la proporción de víctimas no residentes (Spearman 0.71, $p = 0.002$), mientras que la $R$ de encuesta no lo hace (0.16). Como las carpetas y la victimización se cuentan por lugar del hecho, la población flotante se cancela en el cociente; el patrón indica que el registro de las alcaldías centrales incluye víctimas fuera del universo de la encuesta o asigna la alcaldía del hecho con sesgo hacia donde se abre la carpeta.

Por último, la victimización se contrastó con la ENSU, independiente de la ENVIPE, construyendo la misma medida: residencia, seis delitos comunes y nueve ediciones que cubren 2020–2024. Las dos fuentes coinciden en que la victimización varía entre alcaldías en magnitud comparable (razón máx/mín de 1.9 en la ENSU y 2.3 en la ENVIPE), pero no en el orden de las alcaldías (Spearman 0.29, IC 95%: 0.04 a 0.51, frente a un máximo posible de 0.90 dada la fiabilidad de ambas medidas). La discrepancia se atribuye a diferencias de construcción (prevalencia de hogares frente a incidencia, unidad de víctima y periodo de recuerdo) y no afecta el resultado sobre $R$, que la ENSU no mide.